In [1]:
!pip install -q \
llama-index \
chromadb \
sentence-transformers \
rank-bm25 \
transformers \
accelerate \
langchain \
langgraph \
ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 23.8 MB/s eta 0:00:00


In [2]:
import chromadb
import torch
import transformers
import langgraph

print("All libraries installed successfully!")

print("GPU Available:", torch.cuda.is_available())

All libraries installed successfully!
GPU Available: True


In [3]:
!pip uninstall -y datasets
!pip install datasets==2.18.0 -q

Found existing installation: datasets 5.0.1
Uninstalling datasets-5.0.1:
  Successfully uninstalled datasets-5.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ragas 0.4.3 requires datasets>=4.0.0, but you have datasets 2.18.0 which is incompatible.


restart runtime


In [4]:
import datasets

print("Datasets version:", datasets.__version__)

Datasets version: 2.18.0


In [5]:
from datasets import load_dataset

claims_dataset = load_dataset(
    "allenai/scifact",
    "claims",
    trust_remote_code=True
)

print(claims_dataset)

Generating train split:   0%|          | 0/1261 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/450 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'],
        num_rows: 1261
    })
    test: Dataset({
        features: ['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'],
        num_rows: 300
    })
    validation: Dataset({
        features: ['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'],
        num_rows: 450
    })
})


In [6]:
from datasets import load_dataset

claims_dataset = load_dataset(
    "allenai/scifact",
    "claims"
)

corpus_dataset = load_dataset(
    "allenai/scifact",
    "corpus"
)

print(claims_dataset)
print(corpus_dataset)

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1461: FutureWarning: The repository for allenai/scifact contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/allenai/scifact
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/5183 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'],
        num_rows: 1261
    })
    test: Dataset({
        features: ['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'],
        num_rows: 300
    })
    validation: Dataset({
        features: ['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'],
        num_rows: 450
    })
})
DatasetDict({
    train: Dataset({
        features: ['doc_id', 'title', 'abstract', 'structured'],
        num_rows: 5183
    })
})


In [7]:
print(claims_dataset["train"][0])

{'id': 0, 'claim': '0-dimensional biomaterials lack inductive properties.', 'evidence_doc_id': '', 'evidence_label': '', 'evidence_sentences': [], 'cited_doc_ids': [31715818]}


In [8]:
valid_claims = []

claims_train = claims_dataset["train"]

for item in claims_train:

    if item["evidence_label"] != "":
        valid_claims.append(item)

print("Valid claims:", len(valid_claims))

print("\nSample valid claim:\n")
print(valid_claims[0])

Valid claims: 957

Sample valid claim:

{'id': 2, 'claim': '1 in 5 million in UK have abnormal PrP positivity.', 'evidence_doc_id': '13734012', 'evidence_label': 'CONTRADICT', 'evidence_sentences': [4], 'cited_doc_ids': [13734012]}


In [9]:
print(corpus_dataset["train"][0])

{'doc_id': 4983, 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'abstract': ['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.', 'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).', 'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.', 'In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms.', 'In the posterior limb of the int

In [10]:
documents = []

corpus_train = corpus_dataset["train"]

for i in range(len(corpus_train)):

    title = corpus_train[i]["title"]

    abstract_sentences = corpus_train[i]["abstract"]

    abstract_text = " ".join(abstract_sentences)

    full_text = title + " " + abstract_text

    documents.append({
        "doc_id": corpus_train[i]["doc_id"],
        "text": full_text
    })

print("Total documents:", len(documents))

print("\nSample document:\n")

print(documents[0])

Total documents: 5183

Sample document:

{'doc_id': 4983, 'text': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior lim

In [11]:
all_documents = documents

print("Subset size:", len(all_documents))

Subset size: 5183


In [12]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [13]:
texts = [doc["text"] for doc in documents]

print("Documents:", len(texts))

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

print("Embedding shape:", embeddings.shape)

Documents: 5183


Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Embedding shape: (5183, 384)


Semantic retrieval



In [14]:
import chromadb

# Create Chroma client
client = chromadb.Client()

# Create collection
collection = client.create_collection(
    name="scifact_collection"
)

print("ChromaDB collection created!")

ChromaDB collection created!


In [15]:
ids = [str(i) for i in range(len(documents))]

documents_text = [doc["text"] for doc in documents]

metadatas = [
    {"doc_id": str(doc["doc_id"])}
    for doc in documents
]

collection.add(
    ids=ids,
    documents=documents_text,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

In [16]:
query = "Does a biomaterial improve tissue regeneration?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(results)

{'ids': [['4908', '1044', '1595']], 'embeddings': None, 'documents': [['Cell and biomolecule delivery for tissue repair and regeneration in the central nervous system. Tissue engineering frequently involves cells and scaffolds to replace damaged or diseased tissue. It originated, in part, as a means of effecting the delivery of biomolecules such as insulin or neurotrophic factors, given that cells are constitutive producers of such therapeutic agents. Thus cell delivery is intrinsic to tissue engineering. Controlled release of biomolecules is also an important tool for enabling cell delivery since the biomolecules can enable cell engraftment, modulate inflammatory response or otherwise benefit the behavior of the delivered cells. We describe advances in cell and biomolecule delivery for tissue regeneration, with emphasis on the central nervous system (CNS). In the first section, the focus is on encapsulated cell therapy. In the second section, the focus is on biomolecule delivery in po

In [17]:
retrieved_docs = results["documents"][0]

for i, doc in enumerate(retrieved_docs):

    print(f"\n========== DOCUMENT {i+1} ==========\n")

    print(doc[:1000])


========== DOCUMENT 1 ==========

Cell and biomolecule delivery for tissue repair and regeneration in the central nervous system. Tissue engineering frequently involves cells and scaffolds to replace damaged or diseased tissue. It originated, in part, as a means of effecting the delivery of biomolecules such as insulin or neurotrophic factors, given that cells are constitutive producers of such therapeutic agents. Thus cell delivery is intrinsic to tissue engineering. Controlled release of biomolecules is also an important tool for enabling cell delivery since the biomolecules can enable cell engraftment, modulate inflammatory response or otherwise benefit the behavior of the delivered cells. We describe advances in cell and biomolecule delivery for tissue regeneration, with emphasis on the central nervous system (CNS). In the first section, the focus is on encapsulated cell therapy. In the second section, the focus is on biomolecule delivery in polymeric nano/microspheres and hydroge

In [18]:
context = "\n\n".join(retrieved_docs)

prompt = f"""
You are a scientific summarization assistant.

Using ONLY the retrieved documents below,
generate a short factual summary.

Documents:
{context}

Summary:
"""

print(prompt[:3000])


You are a scientific summarization assistant.

Using ONLY the retrieved documents below,
generate a short factual summary.

Documents:
Cell and biomolecule delivery for tissue repair and regeneration in the central nervous system. Tissue engineering frequently involves cells and scaffolds to replace damaged or diseased tissue. It originated, in part, as a means of effecting the delivery of biomolecules such as insulin or neurotrophic factors, given that cells are constitutive producers of such therapeutic agents. Thus cell delivery is intrinsic to tissue engineering. Controlled release of biomolecules is also an important tool for enabling cell delivery since the biomolecules can enable cell engraftment, modulate inflammatory response or otherwise benefit the behavior of the delivered cells. We describe advances in cell and biomolecule delivery for tissue regeneration, with emphasis on the central nervous system (CNS). In the first section, the focus is on encapsulated cell therapy. I

In [19]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 loaded successfully!


In [20]:
inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

outputs = model.generate(
    **inputs,
    max_new_tokens=128
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(result)

We describe advances in cell and biomolecule delivery for tissue regeneration, with emphasis on the central nervous system.


Prosecutor

In [21]:
generated_summary = result

print("Generated Summary:\n")
print(generated_summary)

Generated Summary:

We describe advances in cell and biomolecule delivery for tissue regeneration, with emphasis on the central nervous system.


In [22]:
prosecutor_prompt = f"""
You are a scientific fact-checking prosecutor.

Your task is to identify whether the following summary
contains unsupported claims, contradictions,
or possible scope mismatches.

Summary:
{generated_summary}

Retrieved Evidence:
{context}

Classify the result into ONE of:
- AGREE
- CONTRADICT
- SCOPE-MISMATCH

Also explain WHY.
"""

print(prosecutor_prompt[:3000])


You are a scientific fact-checking prosecutor.

Your task is to identify whether the following summary
contains unsupported claims, contradictions,
or possible scope mismatches.

Summary:
We describe advances in cell and biomolecule delivery for tissue regeneration, with emphasis on the central nervous system.

Retrieved Evidence:
Cell and biomolecule delivery for tissue repair and regeneration in the central nervous system. Tissue engineering frequently involves cells and scaffolds to replace damaged or diseased tissue. It originated, in part, as a means of effecting the delivery of biomolecules such as insulin or neurotrophic factors, given that cells are constitutive producers of such therapeutic agents. Thus cell delivery is intrinsic to tissue engineering. Controlled release of biomolecules is also an important tool for enabling cell delivery since the biomolecules can enable cell engraftment, modulate inflammatory response or otherwise benefit the behavior of the delivered cells

In [23]:
inputs = tokenizer(
    prosecutor_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=128
)

prosecutor_result = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(prosecutor_result)

SCOPE-MISMATCH


In [24]:
matrix_prompt = f"""
You are a contradiction classification system.

Analyze the relationship between the summary
and the retrieved evidence.

Summary:
{generated_summary}

Evidence:
{context}

Return ONLY one label:

AGREE
CONTRADICT
SCOPE-MISMATCH
"""

print(matrix_prompt[:3000])


You are a contradiction classification system.

Analyze the relationship between the summary
and the retrieved evidence.

Summary:
We describe advances in cell and biomolecule delivery for tissue regeneration, with emphasis on the central nervous system.

Evidence:
Cell and biomolecule delivery for tissue repair and regeneration in the central nervous system. Tissue engineering frequently involves cells and scaffolds to replace damaged or diseased tissue. It originated, in part, as a means of effecting the delivery of biomolecules such as insulin or neurotrophic factors, given that cells are constitutive producers of such therapeutic agents. Thus cell delivery is intrinsic to tissue engineering. Controlled release of biomolecules is also an important tool for enabling cell delivery since the biomolecules can enable cell engraftment, modulate inflammatory response or otherwise benefit the behavior of the delivered cells. We describe advances in cell and biomolecule delivery for tissue 

In [25]:
inputs = tokenizer(
    matrix_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=20
)

matrix_result = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Matrix Classification:")
print(matrix_result)

Matrix Classification:
SCOPE-MISMATCH


ADD BM25 hybrid retrieval

In [26]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [
    doc["text"].lower().split()
    for doc in all_documents
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 initialized!")

BM25 initialized!


In [27]:
query = "Does biomaterial improve tissue regeneration?"

tokenized_query = query.lower().split()

bm25_results = bm25.get_top_n(
    tokenized_query,
    all_documents,
    n=3
)

for i, doc in enumerate(bm25_results):

    print(f"\n===== BM25 RESULT {i+1} =====\n")

    print(doc["text"][:1000])


===== BM25 RESULT 1 =====

The Drosophila hindgut lacks constitutively active adult stem cells but proliferates in response to tissue damage. The adult Drosophila hindgut was recently reported to contain active, tissue-replenishing stem cells, like those of the midgut, but located within an anterior ring so as to comprise a single giant crypt. In contrast to this view, we observed no active stem cells and little cell turnover in adult hindgut tissue based on clonal marking and BrdU incorporation studies. Again contradicting the previous proposal, we showed that the adult hindgut is not generated by anterior stem cells during larval/pupal development. However, severe tissue damage within the hindgut elicits cell proliferation within a ring of putative quiescent stem cells at the anterior of the pylorus. Thus, the hindgut does not provide a model of tissue maintenance by constitutively active stem cells, but has great potential to illuminate mechanisms of stress-induced tissue repair.



Combine Semantic + BM25 Retrieval

In [28]:
query = "Does biomaterial improve tissue regeneration?"

query_embedding = embedding_model.encode(query).tolist()

semantic_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

semantic_docs = semantic_results["documents"][0]

print("Semantic Retrieval Ready!")

Semantic Retrieval Ready!


In [29]:
tokenized_query = query.lower().split()

bm25_results = bm25.get_top_n(
    tokenized_query,
    all_documents,
    n=3
)

print("BM25 Retrieval Ready!")

BM25 Retrieval Ready!


In [30]:
hybrid_docs = []

# Add semantic docs
for doc in semantic_docs:
    hybrid_docs.append(doc)

# Add BM25 docs
for doc in bm25_results:

    if doc["text"] not in hybrid_docs:
        hybrid_docs.append(doc["text"])

print("Total Hybrid Documents:", len(hybrid_docs))

Total Hybrid Documents: 6


In [31]:
for i, doc in enumerate(hybrid_docs):

    print(f"\n========== HYBRID DOC {i+1} ==========\n")

    print(doc[:1000])


========== HYBRID DOC 1 ==========

Cell and biomolecule delivery for tissue repair and regeneration in the central nervous system. Tissue engineering frequently involves cells and scaffolds to replace damaged or diseased tissue. It originated, in part, as a means of effecting the delivery of biomolecules such as insulin or neurotrophic factors, given that cells are constitutive producers of such therapeutic agents. Thus cell delivery is intrinsic to tissue engineering. Controlled release of biomolecules is also an important tool for enabling cell delivery since the biomolecules can enable cell engraftment, modulate inflammatory response or otherwise benefit the behavior of the delivered cells. We describe advances in cell and biomolecule delivery for tissue regeneration, with emphasis on the central nervous system (CNS). In the first section, the focus is on encapsulated cell therapy. In the second section, the focus is on biomolecule delivery in polymeric nano/microspheres and hydro

In [32]:
hybrid_context = "\n\n".join(hybrid_docs)

print(hybrid_context[:3000])

Cell and biomolecule delivery for tissue repair and regeneration in the central nervous system. Tissue engineering frequently involves cells and scaffolds to replace damaged or diseased tissue. It originated, in part, as a means of effecting the delivery of biomolecules such as insulin or neurotrophic factors, given that cells are constitutive producers of such therapeutic agents. Thus cell delivery is intrinsic to tissue engineering. Controlled release of biomolecules is also an important tool for enabling cell delivery since the biomolecules can enable cell engraftment, modulate inflammatory response or otherwise benefit the behavior of the delivered cells. We describe advances in cell and biomolecule delivery for tissue regeneration, with emphasis on the central nervous system (CNS). In the first section, the focus is on encapsulated cell therapy. In the second section, the focus is on biomolecule delivery in polymeric nano/microspheres and hydrogels for the nerve regeneration and e

In [33]:
hybrid_prompt = f"""
You are a scientific summarization assistant.

Using ONLY the retrieved evidence below,
generate a concise factual summary.

Evidence:
{hybrid_context}

Summary:
"""

In [34]:
inputs = tokenizer(
    hybrid_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=128
)

improved_summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(improved_summary)

Adult Drosophila hindgut contains active, tissue-replenishing stem cells, but is located within an anterior ring so as to comprise a ring.


real claim



In [35]:
real_claim = valid_claims[0]["claim"]

print(real_claim)

1 in 5 million in UK have abnormal PrP positivity.


In [36]:
query = real_claim

query_embedding = embedding_model.encode(query).tolist()

semantic_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

semantic_docs = semantic_results["documents"][0]

print("semantic Retrieved docs ready!")

semantic Retrieved docs ready!


In [37]:
tokenized_query = query.lower().split()

bm25_results = bm25.get_top_n(
    tokenized_query,
    all_documents,
    n=3
)

hybrid_docs = []

for doc in semantic_docs:
    hybrid_docs.append(doc)

for doc in bm25_results:

    if doc["text"] not in hybrid_docs:
        hybrid_docs.append(doc["text"])

print("Hybrid retrieval completed!")

Hybrid retrieval completed!


install cross encoder

In [38]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded successfully!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded successfully!


In [39]:
pairs = [
    (query, doc)
    for doc in hybrid_docs
]

scores = reranker.predict(pairs)

ranked_docs = sorted(
    zip(hybrid_docs, scores),
    key=lambda x: x[1],
    reverse=True
)

top_docs = [
    ranked_docs[i][0]
    for i in range(min(2, len(ranked_docs)))
]

reranked_context = "\n\n".join(top_docs)

print(reranked_context[:3000])

Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   
 DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   
 SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospitals in regions with lower levels of participation in that survey.    SAMPLE 32,441 archived appendix samples fixed in formalin and embedded in paraffin and tested for the presence of abnormal prion prot

In [40]:
reranked_prompt = f"""
You are a scientific summarization assistant.

Using ONLY the evidence below,
generate a concise factual summary.

Evidence:
{reranked_context}

Summary:
"""

In [41]:
inputs = tokenizer(
    reranked_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=128
)

reranked_summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(reranked_summary)

High body-mass index is associated with 22 common cancers, but its effects vary by site.


In [42]:
strict_prompt = f"""
Classify the relationship between the SUMMARY and EVIDENCE.

Possible labels:
AGREE
CONTRADICT
SCOPE-MISMATCH

SUMMARY:
{reranked_summary}

EVIDENCE:
{reranked_context}

Return ONLY ONE LABEL.
"""

In [43]:
inputs = tokenizer(
    strict_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=10
)

strict_result = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Classification:")
print(strict_result)

Classification:
SCOPE-MISMATCH


actual cited evidence

In [44]:
real_claim_item = valid_claims[0]

print(real_claim_item)

{'id': 2, 'claim': '1 in 5 million in UK have abnormal PrP positivity.', 'evidence_doc_id': '13734012', 'evidence_label': 'CONTRADICT', 'evidence_sentences': [4], 'cited_doc_ids': [13734012]}


In [45]:
gold_doc_ids = real_claim_item["cited_doc_ids"]

gold_documents = []

for doc in documents:

    if doc["doc_id"] in gold_doc_ids:
        gold_documents.append(doc["text"])

print("Gold documents found:", len(gold_documents))

Gold documents found: 1


In [46]:
for i, doc in enumerate(gold_documents):

    print(f"\n===== GOLD DOC {i+1} =====\n")

    print(doc[:1500])


===== GOLD DOC 1 =====

Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   
 DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   
 SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospitals in regions with lower levels of participation in that survey.    SAMPLE 32,441 archived appendix samples fixed in formalin and embedded in paraffin and tested for the presen

In [47]:
gold_context = "\n\n".join(gold_documents)

print(gold_context[:3000])

Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   
 DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   
 SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospitals in regions with lower levels of participation in that survey.    SAMPLE 32,441 archived appendix samples fixed in formalin and embedded in paraffin and tested for the presence of abnormal prion prot

In [48]:
gold_prompt = f"""
Using ONLY the evidence below,
generate a factual scientific summary.

Evidence:
{gold_context}

Summary:
"""

In [49]:
inputs = tokenizer(
    gold_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=128
)

gold_summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(gold_summary)

A large scale survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.


In [50]:
print("\n=========== ORIGINAL CLAIM ===========\n")
print(real_claim)

print("\n=========== GENERATED SUMMARY ===========\n")
print(gold_summary)

print("\n=========== GOLD EVIDENCE ===========\n")
print(gold_context[:3000])


=========== ORIGINAL CLAIM ===========

1 in 5 million in UK have abnormal PrP positivity.

=========== GENERATED SUMMARY ===========

A large scale survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.

=========== GOLD EVIDENCE ===========

Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was a

In [51]:
gold_prosecutor_prompt = f"""
Classify the relationship between the SUMMARY and EVIDENCE.

Possible labels:
AGREE
CONTRADICT
SCOPE-MISMATCH

SUMMARY:
{gold_summary}

EVIDENCE:
{gold_context}

Return ONLY ONE LABEL.
"""

In [52]:
inputs = tokenizer(
    gold_prosecutor_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=10
)

gold_alignment = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Alignment Result:")
print(gold_alignment)

Alignment Result:
SCOPE-MISMATCH


In [53]:
print("CLAIM:\n")
print(real_claim)

print("\nSUMMARY:\n")
print(gold_summary)

CLAIM:

1 in 5 million in UK have abnormal PrP positivity.

SUMMARY:

A large scale survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.


In [54]:
agree_summary = """
A large scale survey was conducted to estimate
the prevalence of subclinical prion infection
in the UK population after the bovine spongiform
encephalopathy epidemic.
"""

print(agree_summary)


A large scale survey was conducted to estimate
the prevalence of subclinical prion infection
in the UK population after the bovine spongiform
encephalopathy epidemic.



In [55]:
agree_prompt = f"""
Classify the relationship between SUMMARY and EVIDENCE.

Possible labels:
AGREE
CONTRADICT
SCOPE-MISMATCH

SUMMARY:
{agree_summary}

EVIDENCE:
{gold_context}

Return ONLY ONE LABEL. and the reason why.
"""

In [56]:
inputs = tokenizer(
    agree_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=10
)

agree_result = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Classification:")
print(agree_result)

Classification:
SCOPE-MISMATCH


the FLAN-T5 model is bottleneck for prescutor, so we keep it as synthesiser
  

DeBERTa NLI model as prosecutor

In [57]:
from transformers import pipeline

nli_classifier = pipeline(
    "text-classification",
    model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0",
    device=0
)

print("NLI classifier loaded!")

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  870MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

NLI classifier loaded!


In [58]:
premise = gold_context[:2000]

hypothesis = agree_summary

print("Premise Ready")
print("Hypothesis Ready")

Premise Ready
Hypothesis Ready


In [59]:
result = nli_classifier(
    f"{premise} [SEP] {hypothesis}"
)

print(result)

[{'label': 'entailment', 'score': 0.9757392406463623}]


scenario- contradict

In [60]:
contradict_summary = """
No evidence of abnormal prion positivity
was found in the UK population.
"""

print(contradict_summary)


No evidence of abnormal prion positivity
was found in the UK population.



In [61]:
premise = gold_context[:2000]

hypothesis = contradict_summary

result = nli_classifier(
    f"{premise} [SEP] {hypothesis}"
)

print(result)

[{'label': 'not_entailment', 'score': 0.9998036026954651}]


mismatch scenario

In [62]:
scope_summary = """
Prion infections may affect broader
population groups beyond previously
studied cohorts.
"""

print(scope_summary)


Prion infections may affect broader
population groups beyond previously
studied cohorts.



In [63]:
hypothesis = scope_summary

result = nli_classifier(
    f"{premise} [SEP] {hypothesis}"
)

print(result)

[{'label': 'not_entailment', 'score': 0.8734309077262878}]


In [64]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

nli_model_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

nli_tokenizer = AutoTokenizer.from_pretrained(
    nli_model_name
)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    nli_model_name
)

print("NLI model reloaded!")
print(nli_model.config.id2label)

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

NLI model reloaded!
{0: 'entailment', 1: 'neutral', 2: 'contradiction'}


In [65]:
print(nli_model.config.id2label)

{0: 'entailment', 1: 'neutral', 2: 'contradiction'}


In [66]:
def classify_nli(premise, hypothesis):

    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():

        outputs = nli_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    predicted_class = torch.argmax(probs).item()

    predicted_label = nli_model.config.id2label[
        predicted_class
    ]

    return {
        "label": predicted_label,
        "scores": probs.tolist()[0]
    }

In [67]:
print(classify_nli(
    "Cats are animals.",
    "Cats are animals."
))

print(classify_nli(
    "Cats are animals.",
    "Cats are plants."
))

print(classify_nli(
    "The sky is blue.",
    "The sky has a color."
))

{'label': 'entailment', 'scores': [0.998046875, 0.0017747879028320312, 0.00034880638122558594]}
{'label': 'contradiction', 'scores': [0.00013077259063720703, 0.0003540515899658203, 0.99951171875]}
{'label': 'entailment', 'scores': [0.99267578125, 0.00646209716796875, 0.0006198883056640625]}


In [68]:
result = classify_nli(
    gold_context[:2000],
    agree_summary
)


print(result)

{'label': 'entailment', 'scores': [0.95703125, 0.01401519775390625, 0.0289154052734375]}


In [69]:
result = classify_nli(
    gold_context[:2000],
    contradict_summary
)

print(result)

{'label': 'contradiction', 'scores': [0.0015316009521484375, 0.001125335693359375, 0.99755859375]}


In [70]:
focused_premise = """
A large scale survey of archived appendix samples
was conducted to estimate the prevalence of
subclinical prion infection in the UK population.
"""

print(focused_premise)


A large scale survey of archived appendix samples
was conducted to estimate the prevalence of
subclinical prion infection in the UK population.



In [71]:
better_scope_summary = """
The survey investigated prevalence patterns
of subclinical prion infection in selected
population cohorts.
"""

print(better_scope_summary)


The survey investigated prevalence patterns
of subclinical prion infection in selected
population cohorts.



In [72]:
result = classify_nli(
    focused_premise,
    better_scope_summary
)

print(result)

{'label': 'contradiction', 'scores': [0.0303192138671875, 0.284912109375, 0.6845703125]}


In [73]:
result = classify_nli(
    focused_premise,
    contradict_summary
)

print(result)

{'label': 'contradiction', 'scores': [0.005947113037109375, 0.445068359375, 0.548828125]}


In [74]:
result = classify_nli(
    focused_premise,
    agree_summary
)

print(result)

{'label': 'neutral', 'scores': [0.0006275177001953125, 0.9990234375, 0.0004286766052246094]}


irrevlevant scenario

In [75]:
mouse_summary = """
Swim training suppresses tumor growth in mice.
"""

print(mouse_summary)


Swim training suppresses tumor growth in mice.



In [76]:
hypothesis = mouse_summary

result = classify_nli(
    gold_context[:2000],
    hypothesis
)

print(result)

{'label': 'neutral', 'scores': [0.300537109375, 0.59814453125, 0.10162353515625]}


hallucinated


In [77]:
hallucinated_summary = """
50% of the UK population showed
abnormal PrP positivity.
"""

print(hallucinated_summary)


50% of the UK population showed
abnormal PrP positivity.



In [78]:
result = classify_nli(
    focused_premise,
    scope_summary
)

print(result)

{'label': 'neutral', 'scores': [0.02410888671875, 0.97509765625, 0.0005807876586914062]}


In [79]:
hypothesis = hallucinated_summary

result = nli_classifier(
    f"{premise} [SEP] {hypothesis}"
)

print(result)

[{'label': 'not_entailment', 'score': 0.9976993203163147}]


Does a biomaterial improve tissue regeneration?" query

In [80]:
bio_query = "Does a biomaterial improve tissue regeneration?"

print(bio_query)

Does a biomaterial improve tissue regeneration?


In [81]:
query_embedding = embedding_model.encode(
    bio_query
).tolist()

semantic_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

semantic_docs = semantic_results["documents"][0]

print("Semantic retrieval completed!")

Semantic retrieval completed!


In [82]:
tokenized_query = bio_query.lower().split()

bm25_results = bm25.get_top_n(
    tokenized_query,
    all_documents,
    n=5
)

print("BM25 retrieval completed!")

BM25 retrieval completed!


In [83]:
hybrid_docs = []

for doc in semantic_docs:
    hybrid_docs.append(doc)

for doc in bm25_results:

    if doc["text"] not in hybrid_docs:
        hybrid_docs.append(doc["text"])

print("Hybrid docs:", len(hybrid_docs))

Hybrid docs: 10


In [84]:
pairs = [
    (bio_query, doc)
    for doc in hybrid_docs
]

scores = reranker.predict(pairs)

ranked_docs = sorted(
    zip(hybrid_docs, scores),
    key=lambda x: x[1],
    reverse=True
)

top_docs = [
    ranked_docs[i][0]
    for i in range(min(2, len(ranked_docs)))
]

bio_context = "\n\n".join(top_docs)

print(bio_context[:3000])

Cell and biomolecule delivery for tissue repair and regeneration in the central nervous system. Tissue engineering frequently involves cells and scaffolds to replace damaged or diseased tissue. It originated, in part, as a means of effecting the delivery of biomolecules such as insulin or neurotrophic factors, given that cells are constitutive producers of such therapeutic agents. Thus cell delivery is intrinsic to tissue engineering. Controlled release of biomolecules is also an important tool for enabling cell delivery since the biomolecules can enable cell engraftment, modulate inflammatory response or otherwise benefit the behavior of the delivered cells. We describe advances in cell and biomolecule delivery for tissue regeneration, with emphasis on the central nervous system (CNS). In the first section, the focus is on encapsulated cell therapy. In the second section, the focus is on biomolecule delivery in polymeric nano/microspheres and hydrogels for the nerve regeneration and e

In [85]:
bio_prompt = f"""
Using ONLY the evidence below,
generate a factual scientific summary.

Evidence:
{bio_context}

Summary:
"""

In [86]:
inputs = tokenizer(
    bio_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=128
)

bio_summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(bio_summary)

Transplantation of human cardiosphere-derived cells into infarcted mice provides a model for regenerative and endogenous regeneration.


In [87]:
premise = bio_context[:2000]

hypothesis = bio_summary

result = classify_nli(
    gold_context[:2000],
    hypothesis
)

print(result)

{'label': 'neutral', 'scores': [0.1942138671875, 0.75, 0.05572509765625]}


In [88]:
def map_label(nli_label):

    if nli_label == "entailment":
        return "AGREE"

    elif nli_label == "contradiction":
        return "CONTRADICT"

    else:
        return "SCOPE-MISMATCH"

In [89]:
print(
    map_label(
        classify_nli(
            "Cats are animals.",
            "Cats are animals."
        )["label"]
    )
)

AGREE


In [90]:
evaluation_results = []

print("Evaluation storage ready!")

Evaluation storage ready!


In [91]:
test_claims = valid_claims[:5]

print("Claims selected:", len(test_claims))

Claims selected: 5


In [92]:
for item in test_claims:

    claim = item["claim"]

    print("\n================================")
    print("CLAIM:")
    print(claim)

    # Semantic retrieval
    query_embedding = embedding_model.encode(
        claim
    ).tolist()

    semantic_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    semantic_docs = semantic_results["documents"][0]

    # BM25 retrieval
    tokenized_query = claim.lower().split()

    bm25_results = bm25.get_top_n(
        tokenized_query,
        all_documents,
        n=3
    )

    # Hybrid retrieval
    hybrid_docs = []

    for doc in semantic_docs:
        hybrid_docs.append(doc)

    for doc in bm25_results:

        if doc["text"] not in hybrid_docs:
            hybrid_docs.append(doc["text"])

    # Reranking
    pairs = [
        (claim, doc)
        for doc in hybrid_docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(hybrid_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    top_docs = [
        ranked_docs[i][0]
        for i in range(min(2, len(ranked_docs)))
    ]

    context = "\n\n".join(top_docs)

    # Summary generation
    prompt = f"""
    CLAIM:
    {claim}

    EVIDENCE:
    {context}

    Using ONLY the evidence above:

    1. Determine whether the claim is supported, contradicted, or partially supported.
    2. Explain the evidence in 1-2 sentences.
    3. Do not introduce information not present in the evidence.

    ANSWER:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=128
    )

    new_answer = tokenizer.decode(
      outputs[0],
      skip_special_tokens=True
    )

    print(new_answer)
    # NLI verification
    result = classify_nli(
        context[:1000],
        new_answer
    )

    final_label = map_label(
        result["label"]
    )

    print("\nSUMMARY:")
    print(new_answer)

    print("\nFINAL LABEL:")
    print(final_label)

    evaluation_results.append({
        "claim": claim,
        "summary": new_answer,
        "label": final_label,
        "nli_scores": result["scores"]
    })


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.
High body-mass index is associated with 22 common cancers, but is not associated with specific site-specific cancers.

SUMMARY:
High body-mass index is associated with 22 common cancers, but is not associated with specific site-specific cancers.

FINAL LABEL:
SCOPE-MISMATCH

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
Noninvasive ventilation is associated with a reduction in endotracheal intubation in recipients of solid organ transplantation with acute respiratory failure.

SUMMARY:
Noninvasive ventilation is associated with a reduction in endotracheal intubation in recipients of solid organ transplantation with acute respiratory failure.

FINAL LABEL:
CONTRADICT

CLAIM:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.
Folic acid supplementation reduced serum homocysteine levels.

SUMMARY:
Fol

with all docs

In [93]:
total_chars = sum(len(doc["text"]) for doc in documents)

print("Documents:", len(documents))
print("Characters:", total_chars)
print("Average chars:", total_chars/len(documents))

Documents: 5183
Characters: 7785948
Average chars: 1502.2087594057496


In [94]:
print("CLAIM")
print(claim)

print("\nTOP DOC 1")
print(top_docs[0][:2000])

print("\nTOP DOC 2")
print(top_docs[1][:2000])

CLAIM
76-85% of people with severe mental disorder receive no treatment in low and middle income countries.

TOP DOC 1
Prevalence, severity, and unmet need for treatment of mental disorders in the World Health Organization World Mental Health Surveys. CONTEXT Little is known about the extent or severity of untreated mental disorders, especially in less-developed countries.   
 OBJECTIVE To estimate prevalence, severity, and treatment of Diagnostic and Statistical Manual of Mental Disorders, Fourth Edition (DSM-IV) mental disorders in 14 countries (6 less developed, 8 developed) in the World Health Organization (WHO) World Mental Health (WMH) Survey Initiative.   
 DESIGN, SETTING, AND PARTICIPANTS Face-to-face household surveys of 60 463 community adults conducted from 2001-2003 in 14 countries in the Americas, Europe, the Middle East, Africa, and Asia.   
 MAIN OUTCOME MEASURES The DSM-IV disorders, severity, and treatment were assessed with the WMH version of the WHO Composite Intern

In [95]:
claim = "76-85% of people with severe mental disorder receive no treatment in low and middle income countries."

query_embedding = embedding_model.encode(
    claim
).tolist()

semantic_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

semantic_docs = semantic_results["documents"][0]

tokenized_query = claim.lower().split()

bm25_results = bm25.get_top_n(
    tokenized_query,
    all_documents,
    n=3
)

hybrid_docs = []

for doc in semantic_docs:
    hybrid_docs.append(doc)

for doc in bm25_results:

    if doc["text"] not in hybrid_docs:
        hybrid_docs.append(doc["text"])

pairs = [
    (claim, doc)
    for doc in hybrid_docs
]

scores = reranker.predict(pairs)

ranked_docs = sorted(
    zip(hybrid_docs, scores),
    key=lambda x: x[1],
    reverse=True
)

top_docs = [
    ranked_docs[i][0]
    for i in range(min(2, len(ranked_docs)))
]

context = "\n\n".join(top_docs)

print("CLAIM:")
print(claim)

print("\n\nCONTEXT:")
print(context[:4000])

CLAIM:
76-85% of people with severe mental disorder receive no treatment in low and middle income countries.


CONTEXT:
Prevalence, severity, and unmet need for treatment of mental disorders in the World Health Organization World Mental Health Surveys. CONTEXT Little is known about the extent or severity of untreated mental disorders, especially in less-developed countries.   
 OBJECTIVE To estimate prevalence, severity, and treatment of Diagnostic and Statistical Manual of Mental Disorders, Fourth Edition (DSM-IV) mental disorders in 14 countries (6 less developed, 8 developed) in the World Health Organization (WHO) World Mental Health (WMH) Survey Initiative.   
 DESIGN, SETTING, AND PARTICIPANTS Face-to-face household surveys of 60 463 community adults conducted from 2001-2003 in 14 countries in the Americas, Europe, the Middle East, Africa, and Asia.   
 MAIN OUTCOME MEASURES The DSM-IV disorders, severity, and treatment were assessed with the WMH version of the WHO Composite Inter

In [96]:
prompt = f"""
CLAIM:
{claim}

EVIDENCE:
{context}

Using ONLY the evidence above.

Return EXACTLY in this format:

VERDICT: SUPPORTED / CONTRADICTED / PARTIALLY_SUPPORTED

REASON:
One sentence explaining why.

ANSWER:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=128
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

The prevalence of DSM-IV disorders in 14 countries in the World Mental Health Survey Initiative.


conclusion- FLAN-T5 small is bottlenek and not a good synthesiser

now we try FLAN-T5 large

In [97]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

large_model_name = "google/flan-t5-large"

large_tokenizer = AutoTokenizer.from_pretrained(
    large_model_name
)

large_model = AutoModelForSeq2SeqLM.from_pretrained(
    large_model_name
)

print("FLAN-T5 Large loaded!")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 Large loaded!


In [98]:
print(claim)
print("\n")
print(context[:1000])

76-85% of people with severe mental disorder receive no treatment in low and middle income countries.


Prevalence, severity, and unmet need for treatment of mental disorders in the World Health Organization World Mental Health Surveys. CONTEXT Little is known about the extent or severity of untreated mental disorders, especially in less-developed countries.   
 OBJECTIVE To estimate prevalence, severity, and treatment of Diagnostic and Statistical Manual of Mental Disorders, Fourth Edition (DSM-IV) mental disorders in 14 countries (6 less developed, 8 developed) in the World Health Organization (WHO) World Mental Health (WMH) Survey Initiative.   
 DESIGN, SETTING, AND PARTICIPANTS Face-to-face household surveys of 60 463 community adults conducted from 2001-2003 in 14 countries in the Americas, Europe, the Middle East, Africa, and Asia.   
 MAIN OUTCOME MEASURES The DSM-IV disorders, severity, and treatment were assessed with the WMH version of the WHO Composite International Diagnos

In [99]:
prompt = f"""
CLAIM:
{claim}

EVIDENCE:
{context}

Using ONLY the evidence above.

Return EXACTLY in this format:

VERDICT: SUPPORTED / CONTRADICTED / PARTIALLY_SUPPORTED

REASON:
One sentence explaining why.

ANSWER:
"""
inputs = large_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = large_model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False
)

large_answer = large_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(large_answer)

SUPPORTED


In [100]:
for item in test_claims:

    claim = item["claim"]

    print("\n================================")
    print("CLAIM:")
    print(claim)

    # Semantic retrieval
    query_embedding = embedding_model.encode(
        claim
    ).tolist()

    semantic_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    semantic_docs = semantic_results["documents"][0]

    # BM25 retrieval
    tokenized_query = claim.lower().split()

    bm25_results = bm25.get_top_n(
        tokenized_query,
        all_documents,
        n=3
    )

    # Hybrid retrieval
    hybrid_docs = []

    for doc in semantic_docs:
        hybrid_docs.append(doc)

    for doc in bm25_results:

        if doc["text"] not in hybrid_docs:
            hybrid_docs.append(doc["text"])

    # Reranking
    pairs = [
        (claim, doc)
        for doc in hybrid_docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(hybrid_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    top_docs = [
        ranked_docs[i][0]
        for i in range(min(2, len(ranked_docs)))
    ]

    context = "\n\n".join(top_docs)

    # Claim-aware verdict generation

    prompt = f"""
    CLAIM:
    {claim}

    EVIDENCE:
    {context}

    Using ONLY the evidence above.

    Return EXACTLY in this format:

    VERDICT: SUPPORTED / CONTRADICTED / PARTIALLY_SUPPORTED

    REASON:
    One sentence explaining why.

    ANSWER:
    """

    inputs = large_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = large_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

    new_answer = large_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("\nANSWER:")
    print(new_answer)

    # Prosecutor checks claim vs generated answer

    result = classify_nli(
        claim,
        new_answer
    )

    final_label = map_label(
        result["label"]
    )

    print("\nFINAL LABEL:")
    print(final_label)

    evaluation_results.append({
        "claim": claim,
        "answer": new_answer,
        "label": final_label,
        "nli_scores": result["scores"]
    })


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

ANSWER:
BMI was associated with 17 of 22 cancers, but effects varied considerably by site.

FINAL LABEL:
SCOPE-MISMATCH

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

ANSWER:
SUPPORTED

FINAL LABEL:
SCOPE-MISMATCH

CLAIM:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

ANSWER:
lowering effect was greatest.

FINAL LABEL:
CONTRADICT

CLAIM:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

ANSWER:
lowering effect was greatest.

FINAL LABEL:
CONTRADICT

CLAIM:
76-85% of people with severe mental disorder receive no treatment in low and middle income countries.

ANSWER:
SUPPORTED

FINAL LABEL:
SCOPE-MISMATCH


In [101]:
claim = "76-85% of people with severe mental disorder receive no treatment in low and middle income countries."

context = top_docs[0]

prompt = f"""
Claim: {claim}

Evidence:
{context}

Question:
Does the evidence support the claim?

Answer in one sentence.
"""

inputs = large_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = large_model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False
)

print(
    large_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Yes


FLAN-T5 large is better but weak.

Mistral-7B-Instruct-v0.2

In [102]:
print(top_docs[0][:3000])

Prevalence, severity, and unmet need for treatment of mental disorders in the World Health Organization World Mental Health Surveys. CONTEXT Little is known about the extent or severity of untreated mental disorders, especially in less-developed countries.   
 OBJECTIVE To estimate prevalence, severity, and treatment of Diagnostic and Statistical Manual of Mental Disorders, Fourth Edition (DSM-IV) mental disorders in 14 countries (6 less developed, 8 developed) in the World Health Organization (WHO) World Mental Health (WMH) Survey Initiative.   
 DESIGN, SETTING, AND PARTICIPANTS Face-to-face household surveys of 60 463 community adults conducted from 2001-2003 in 14 countries in the Americas, Europe, the Middle East, Africa, and Asia.   
 MAIN OUTCOME MEASURES The DSM-IV disorders, severity, and treatment were assessed with the WMH version of the WHO Composite International Diagnostic Interview (WMH-CIDI), a fully structured, lay-administered psychiatric diagnostic interview.   
 RES

In [103]:
print(scores)

[ 2.899505   -0.17460163 -6.5680566  -2.8993626  -0.8311326 ]


In [104]:
import torch
print(torch.cuda.get_device_name(0))

Tesla T4


In [105]:
!nvidia-smi

Thu Aug 20 17:29:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             25W /   70W |    1297MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [106]:
!pip install -q bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.8 MB/s eta 0:00:00


In [107]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

mistral_tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

mistral_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Mistral loaded!")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Mistral loaded!


In [108]:
!nvidia-smi

Thu Aug 20 17:40:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             25W /   70W |    5221MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [109]:
prompt = f"""
Claim:
{claim}

Evidence:
{context}

Task:
Determine whether the claim is:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Explain your reasoning using ONLY the evidence.

Answer format:

VERDICT:
<supported/contradicted/partially_supported>

REASON:
<one or two sentences>
"""

In [110]:
messages = f"<s>[INST] {prompt} [/INST]"

inputs = mistral_tokenizer(
    messages,
    return_tensors="pt"
).to("cuda")

outputs = mistral_model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=mistral_tokenizer.eos_token_id
)

answer = mistral_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

[INST] 
Claim:
76-85% of people with severe mental disorder receive no treatment in low and middle income countries.

Evidence:
Prevalence, severity, and unmet need for treatment of mental disorders in the World Health Organization World Mental Health Surveys. CONTEXT Little is known about the extent or severity of untreated mental disorders, especially in less-developed countries.   
 OBJECTIVE To estimate prevalence, severity, and treatment of Diagnostic and Statistical Manual of Mental Disorders, Fourth Edition (DSM-IV) mental disorders in 14 countries (6 less developed, 8 developed) in the World Health Organization (WHO) World Mental Health (WMH) Survey Initiative.   
 DESIGN, SETTING, AND PARTICIPANTS Face-to-face household surveys of 60 463 community adults conducted from 2001-2003 in 14 countries in the Americas, Europe, the Middle East, Africa, and Asia.   
 MAIN OUTCOME MEASURES The DSM-IV disorders, severity, and treatment were assessed with the WMH version of the WHO Composi

In [111]:
evaluation_results = []

for item in test_claims:

    claim = item["claim"]

    print("\n" + "="*60)
    print("CLAIM:")
    print(claim)

    # =====================================
    # RETRIEVAL
    # =====================================

    query_embedding = embedding_model.encode(
        claim
    ).tolist()

    semantic_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    semantic_docs = semantic_results["documents"][0]

    tokenized_query = claim.lower().split()

    bm25_results = bm25.get_top_n(
        tokenized_query,
        all_documents,
        n=3
    )

    # =====================================
    # HYBRID RETRIEVAL
    # =====================================

    hybrid_docs = []

    for doc in semantic_docs:
        hybrid_docs.append(doc)

    for doc in bm25_results:

        if doc["text"] not in hybrid_docs:
            hybrid_docs.append(doc["text"])

    # =====================================
    # RERANKING
    # =====================================

    pairs = [
        (claim, doc)
        for doc in hybrid_docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(hybrid_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    top_docs = [
        ranked_docs[i][0]
        for i in range(min(2, len(ranked_docs)))
    ]

    context = "\n\n".join(top_docs)

    print("\nTOP EVIDENCE:")
    print(top_docs[0][:500])

    # =====================================
    # MISTRAL SYNTHESIZER
    # =====================================

    prompt = f"""
Claim:
{claim}

Evidence:
{context}

Definitions:

SUPPORTED:
The evidence directly confirms the claim.

CONTRADICTED:
The evidence directly disagrees with the claim.

PARTIALLY_SUPPORTED:
The evidence discusses the same topic but does not fully confirm or deny the claim.

Task:
Choose exactly ONE label.

Output format:

VERDICT: <SUPPORTED, CONTRADICTED, PARTIALLY_SUPPORTED>

REASON: <one sentence>
"""

    messages = f"<s>[INST] {prompt} [/INST]"

    inputs = mistral_tokenizer(
        messages,
        return_tensors="pt"
    ).to("cuda")

    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=mistral_tokenizer.eos_token_id
    )

    answer = mistral_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Remove prompt from output
    answer = answer.split("[/INST]")[-1].strip()

    print("\nANSWER:")
    print(answer)

    # =====================================
    # EXTRACT VERDICT
    # =====================================

    verdict = "UNKNOWN"

    lower_answer = answer.lower()

    if "partially supported" in lower_answer:
        verdict = "PARTIALLY_SUPPORTED"

    elif "supported" in lower_answer:
        verdict = "SUPPORTED"

    elif "contradicted" in lower_answer:
        verdict = "CONTRADICTED"

    print("\nVERDICT:")
    print(verdict)

    evaluation_results.append({
        "claim": claim,
        "top_evidence": top_docs[0][:1000],
        "answer": answer,
        "verdict": verdict,
        "reranker_score": float(scores.max())
    })

print("\nFinished evaluation.")


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

TOP EVIDENCE:
Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood 

ANSWER:
VERDICT: PARTIALLY_SUPPORTED

REASON: The first study provides evidence of the prevalence of abnormal PrP in the UK population, but it does not directly confirm or deny the specific claim of "1 in 5 million" having abnormal PrP positivity.

VERDICT:
SUPPORTED

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

TOP EVIDENCE:
Liver transplantation and opioid 

In [112]:
manual_claims = [

    # SHOULD BE SUPPORTED
    "76-85% of people with severe mental disorder receive no treatment in low and middle income countries.",

    # SHOULD BE CONTRADICTED
    "Less than 5% of people with severe mental disorder receive no treatment in low and middle income countries.",

    # SHOULD BE PARTIALLY SUPPORTED
    "People with mental disorders often do not receive adequate treatment worldwide."
]

# Use the mental health paper directly
context = top_docs[0]

for claim in manual_claims:

    print("\n" + "="*60)
    print("CLAIM:")
    print(claim)

    prompt = f"""
You are a scientific fact checker.

Claim:
{claim}

Evidence:
{context}

Rules:

SUPPORTED:
Evidence directly confirms the claim.

CONTRADICTED:
Evidence directly disagrees with the claim.

PARTIALLY_SUPPORTED:
Evidence discusses the same topic but neither clearly confirms nor clearly contradicts the claim.

You MUST choose the single best label.

If a numerical value in the claim conflicts with a numerical value in the evidence, output CONTRADICTED.

Output ONLY ONE WORD:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED
"""

    messages = f"<s>[INST] {prompt} [/INST]"

    inputs = mistral_tokenizer(
        messages,
        return_tensors="pt"
    ).to("cuda")

    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        temperature=0.0,
        pad_token_id=mistral_tokenizer.eos_token_id
    )

    answer = mistral_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    answer = answer.split("[/INST]")[-1].strip()

    print("\nMODEL OUTPUT:")
    print(answer)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



CLAIM:
76-85% of people with severe mental disorder receive no treatment in low and middle income countries.

MODEL OUTPUT:
PARTIALLY\_SUPPORTED

The

CLAIM:
Less than 5% of people with severe mental disorder receive no treatment in low and middle income countries.

MODEL OUTPUT:
PARTIALLY\_SUPPORTED

The

CLAIM:
People with mental disorders often do not receive adequate treatment worldwide.

MODEL OUTPUT:
PARTIALLY\_SUPPORTED. The evidence


In [113]:
claim1 = "76-85% of people with severe mental disorder receive no treatment in low and middle income countries."

claim2 = "Less than 5% of people with severe mental disorder receive no treatment in low and middle income countries."

evidence = """
Although disorder severity was correlated with probability of treatment in almost all countries,
35.5% to 50.3% of serious cases in developed countries and
76.3% to 85.4% in less-developed countries received no treatment.
"""

print(classify_nli(evidence, claim1))
print(classify_nli(evidence, claim2))

{'label': 'neutral', 'scores': [0.08306884765625, 0.91357421875, 0.0032825469970703125]}
{'label': 'contradiction', 'scores': [0.01971435546875, 0.0213775634765625, 0.958984375]}


In [114]:
print(nli_model.config.id2label)

{0: 'entailment', 1: 'neutral', 2: 'contradiction'}


In [115]:
print(classify_nli(
    "Cats are animals.",
    "Cats are plants."
))

{'label': 'contradiction', 'scores': [0.00013077259063720703, 0.0003540515899658203, 0.99951171875]}


In [116]:
print(type(nli_model))
print(nli_model.config.architectures)
print(nli_model.config.num_labels)

<class 'transformers.models.deberta_v2.modeling_deberta_v2.DebertaV2ForSequenceClassification'>
['DebertaV2ForSequenceClassification']
3


In [117]:
print(nli_model.config.label2id)

{'contradiction': 2, 'entailment': 0, 'neutral': 1}


In [118]:
inputs = nli_tokenizer(
    "Cats are animals.",
    "Cats are plants.",
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = nli_model(**inputs)

print(outputs.logits)

print(torch.softmax(outputs.logits, dim=1))

print(torch.argmax(outputs.logits, dim=1))

tensor([[-3.3398, -2.3438,  5.6016]], dtype=torch.float16)
tensor([[1.3077e-04, 3.5405e-04, 9.9951e-01]], dtype=torch.float16)
tensor([2])


In [119]:
claim = "76-85% of people with severe mental disorder receive no treatment in low and middle income countries."

context = top_docs[0]

print(claim)
print("\n")
print(context[:2000])

76-85% of people with severe mental disorder receive no treatment in low and middle income countries.


Prevalence, severity, and unmet need for treatment of mental disorders in the World Health Organization World Mental Health Surveys. CONTEXT Little is known about the extent or severity of untreated mental disorders, especially in less-developed countries.   
 OBJECTIVE To estimate prevalence, severity, and treatment of Diagnostic and Statistical Manual of Mental Disorders, Fourth Edition (DSM-IV) mental disorders in 14 countries (6 less developed, 8 developed) in the World Health Organization (WHO) World Mental Health (WMH) Survey Initiative.   
 DESIGN, SETTING, AND PARTICIPANTS Face-to-face household surveys of 60 463 community adults conducted from 2001-2003 in 14 countries in the Americas, Europe, the Middle East, Africa, and Asia.   
 MAIN OUTCOME MEASURES The DSM-IV disorders, severity, and treatment were assessed with the WMH version of the WHO Composite International Diagnos

FLAN BASE

In [120]:
prompt = f"""
Claim:
{claim}

Evidence:
{context}

Does the evidence support the claim?

Answer briefly.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

base_answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(base_answer)

The prevalence of DSM-IV disorders in 14 countries in the World Mental Health Survey Initiative varied widely, from 4.3% in Shanghai to 26.4% in the United States, with an interquartile range of 9.1%-16.9%.


FLAN LARGE

In [121]:
inputs = large_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = large_model.generate(
    **inputs,
    max_new_tokens=100
)

large_answer = large_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(large_answer)

The prevalence of mental disorders in the WHO World Mental Health Surveys is high, especially in less-developed countries.


MISTRAL


In [122]:


mistral_prompt = f"""
Claim:
{claim}

Evidence:
{context}

Determine whether the claim is supported.

Explain using the evidence.
"""

messages = f"<s>[INST] {mistral_prompt} [/INST]"

inputs = mistral_tokenizer(
    messages,
    return_tensors="pt"
).to("cuda")

outputs = mistral_model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False,
    pad_token_id=mistral_tokenizer.eos_token_id
)

mistral_answer = mistral_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(mistral_answer)

[INST] 
Claim:
76-85% of people with severe mental disorder receive no treatment in low and middle income countries.

Evidence:
Prevalence, severity, and unmet need for treatment of mental disorders in the World Health Organization World Mental Health Surveys. CONTEXT Little is known about the extent or severity of untreated mental disorders, especially in less-developed countries.   
 OBJECTIVE To estimate prevalence, severity, and treatment of Diagnostic and Statistical Manual of Mental Disorders, Fourth Edition (DSM-IV) mental disorders in 14 countries (6 less developed, 8 developed) in the World Health Organization (WHO) World Mental Health (WMH) Survey Initiative.   
 DESIGN, SETTING, AND PARTICIPANTS Face-to-face household surveys of 60 463 community adults conducted from 2001-2003 in 14 countries in the Americas, Europe, the Middle East, Africa, and Asia.   
 MAIN OUTCOME MEASURES The DSM-IV disorders, severity, and treatment were assessed with the WMH version of the WHO Composi

In [123]:
print("FLAN BASE")
print(
    classify_nli(
        context,
        base_answer
    )
)

print("\nFLAN LARGE")
print(
    classify_nli(
        context,
        large_answer
    )
)

print("\nMISTRAL")
print(
    classify_nli(
        context,
        mistral_answer
    )
)

FLAN BASE
{'label': 'entailment', 'scores': [0.9072265625, 0.05438232421875, 0.038330078125]}

FLAN LARGE
{'label': 'entailment', 'scores': [0.962890625, 0.0341796875, 0.0028076171875]}

MISTRAL
{'label': 'entailment', 'scores': [0.93701171875, 0.0307159423828125, 0.032196044921875]}


In [124]:
selected_claims = [
    valid_claims[0],
    valid_claims[1],
    valid_claims[2]
]

print("Claims selected:")
for i, item in enumerate(selected_claims):
    print(f"\n{i+1}. {item['claim']}")

Claims selected:

1. 1 in 5 million in UK have abnormal PrP positivity.

2. 32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

3. 40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.


In [125]:

def retrieve_evidence(claim):

    query_embedding = embedding_model.encode(
        claim
    ).tolist()

    semantic_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    semantic_docs = semantic_results["documents"][0]

    tokenized_query = claim.lower().split()

    bm25_results = bm25.get_top_n(
        tokenized_query,
        all_documents,
        n=3
    )

    hybrid_docs = []

    for doc in semantic_docs:
        hybrid_docs.append(doc)

    for doc in bm25_results:

        if doc["text"] not in hybrid_docs:
            hybrid_docs.append(doc["text"])

    pairs = [
        (claim, doc)
        for doc in hybrid_docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(hybrid_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    top_docs = [
        ranked_docs[i][0]
        for i in range(min(2, len(ranked_docs)))
    ]

    return "\n\n".join(top_docs)

In [126]:
def run_flan_base(claim, context):

    prompt = f"""
Claim:
{claim}

Evidence:
{context}

Does the evidence support the claim?

Answer briefly.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [127]:
def run_flan_large(claim, context):

    prompt = f"""
Claim:
{claim}

Evidence:
{context}

Does the evidence support the claim?

Answer briefly.
"""

    inputs = large_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = large_model.generate(
        **inputs,
        max_new_tokens=100
    )

    return large_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [128]:
def run_mistral(claim, context):

    prompt = f"""
Claim:
{claim}

Evidence:
{context}

Determine whether the claim is supported.

Explain using the evidence.
"""

    messages = f"<s>[INST] {prompt} [/INST]"

    inputs = mistral_tokenizer(
        messages,
        return_tensors="pt"
    ).to("cuda")

    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=mistral_tokenizer.eos_token_id
    )

    answer = mistral_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [129]:
for idx, item in enumerate(selected_claims):

    claim = item["claim"]

    print("\n" + "="*80)
    print(f"CLAIM {idx+1}")
    print("="*80)

    print(claim)

    context = retrieve_evidence(claim)

    print("\nTOP EVIDENCE:")
    print(context[:800])

    print("\n" + "-"*80)
    print("FLAN-T5 BASE")
    print("-"*80)

    base_answer = run_flan_base(
        claim,
        context
    )

    print(base_answer)

    print("\nNLI:")
    print(
        classify_nli(
            context,
            base_answer
        )
    )

    print("\n" + "-"*80)
    print("FLAN-T5 LARGE")
    print("-"*80)

    large_answer = run_flan_large(
        claim,
        context
    )

    print(large_answer)

    print("\nNLI:")
    print(
        classify_nli(
            context,
            large_answer
        )
    )

    print("\n" + "-"*80)
    print("MISTRAL")
    print("-"*80)

    mistral_answer = run_mistral(
        claim,
        context
    )

    print(mistral_answer)

    print("\nNLI:")
    print(
        classify_nli(
            context,
            mistral_answer
        )
    )


CLAIM 1
1 in 5 million in UK have abnormal PrP positivity.

TOP EVIDENCE:
Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   
 DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   
 SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospital

--------------------------------------------------------------------------------
FLAN-T5 BASE
-----------------------------

In [130]:
print(classify_nli(context, claim))

{'label': 'neutral', 'scores': [0.4326171875, 0.49365234375, 0.0736083984375]}


In [131]:
for idx, item in enumerate(selected_claims):

    claim = item["claim"]

    print("\n" + "="*80)
    print(f"CLAIM {idx+1}")
    print("="*80)

    print(claim)

    context = retrieve_evidence(claim)

    result = classify_nli(
        context[:2000],   # evidence
        claim             # claim
    )

    print("\nNLI RESULT:")
    print(result)

    print("\nFINAL LABEL:")
    print(map_label(result["label"]))


CLAIM 1
1 in 5 million in UK have abnormal PrP positivity.

NLI RESULT:
{'label': 'neutral', 'scores': [0.262451171875, 0.491943359375, 0.2454833984375]}

FINAL LABEL:
SCOPE-MISMATCH

CLAIM 2
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

NLI RESULT:
{'label': 'neutral', 'scores': [0.0928955078125, 0.84619140625, 0.060882568359375]}

FINAL LABEL:
SCOPE-MISMATCH

CLAIM 3
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

NLI RESULT:
{'label': 'entailment', 'scores': [0.56591796875, 0.39013671875, 0.044097900390625]}

FINAL LABEL:
AGREE


In [132]:
for idx, item in enumerate(selected_claims):

    claim = item["claim"]

    context = retrieve_evidence(claim)

    print("\n" + "="*80)
    print(f"CLAIM {idx+1}")
    print("="*80)

    print(claim)

    print("\nTOP EVIDENCE:")
    print(context[:1500])

    result = classify_nli(
        context[:1500],
        claim
    )

    print("\nRESULT:")
    print(result)

    print("\nFINAL:")
    print(map_label(result["label"]))


CLAIM 1
1 in 5 million in UK have abnormal PrP positivity.

TOP EVIDENCE:
Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   
 DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   
 SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospitals in regions with lower levels of participation in that survey.    SAMPLE 32,441 archived appendix samples fixed in formalin 

In [133]:
for idx, item in enumerate(selected_claims):

    claim = item["claim"]

    context = retrieve_evidence(claim)

    print("\n" + "="*80)
    print(claim)

    print("\nFULL TOP DOCUMENT:")
    print(context[:4000])


1 in 5 million in UK have abnormal PrP positivity.

FULL TOP DOCUMENT:
Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   
 DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   
 SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospitals in regions with lower levels of participation in that survey.    SAMPLE 32,441 archived appendix samples fixed in formalin and

In [134]:
print(selected_claims[0].keys())

dict_keys(['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'])


In [135]:
evaluation_results = []

for item in selected_claims:

    claim = item["claim"]

    true_label = item["evidence_label"]

    context = retrieve_evidence(claim)

    result = classify_nli(
        context[:1500],
        claim
    )

    predicted_label = map_label(
        result["label"]
    )

    evaluation_results.append({
        "claim": claim,
        "true_label": true_label,
        "predicted_label": predicted_label
    })

print("Number of evaluated claims:", len(evaluation_results))
print(evaluation_results[0])

Number of evaluated claims: 3
{'claim': '1 in 5 million in UK have abnormal PrP positivity.', 'true_label': 'CONTRADICT', 'predicted_label': 'SCOPE-MISMATCH'}


In [136]:
for row in evaluation_results:

    print("\nCLAIM:")
    print(row["claim"])

    print("TRUE:")
    print(row["true_label"])

    print("PREDICTED:")
    print(row["predicted_label"])


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.
TRUE:
CONTRADICT
PREDICTED:
SCOPE-MISMATCH

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
TRUE:
SUPPORT
PREDICTED:
SCOPE-MISMATCH

CLAIM:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.
TRUE:
SUPPORT
PREDICTED:
SCOPE-MISMATCH


In [137]:
claim = selected_claims[0]["claim"]

print("CLAIM:")
print(claim)

context = retrieve_evidence(claim)

print("\nRETRIEVED EVIDENCE:")
print(context[:3000])

result = classify_nli(
    context[:1500],
    claim
)

print("\nNLI RESULT:")
print(result)

print("\nFINAL LABEL:")
print(map_label(result["label"]))

CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

RETRIEVED EVIDENCE:
Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   
 DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   
 SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospitals in regions with lower levels of participation in that survey.    SAMPLE 32,441 archived appendix samples fixed in forma

In [138]:
claim = selected_claims[1]["claim"]

print("CLAIM:")
print(claim)

context = retrieve_evidence(claim)

print("\nRETRIEVED EVIDENCE:")
print(context[:3000])

result = classify_nli(
    context[:1500],
    claim
)

print("\nNLI RESULT:")
print(result)

print("\nFINAL LABEL:")
print(map_label(result["label"]))

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

RETRIEVED EVIDENCE:
Liver transplantation and opioid dependence. ContextChronic hepatitis C is the leading cause for liver transplantation in the United States. Intravenous drug use, the major risk factor, accounts for approximately 60% of hepatitis C virus transmission. Information from the United Network of Organ Sharing (UNOS) does not address substance use among liver transplantation patients. ObjectiveTo identify addiction-related criteria for admission to the UNOS liver transplantation waiting list and posttransplantation problems experienced by patients who are prescribed maintenance methadone. Design, Setting, and ParticipantsMail survey of all 97 adult US liver transplantation programs (belonging to UNOS) in March 2000 with telephone follow-up conducted in May and June 2000.Main Outcome MeasuresPrograms' acceptance and management of patients with past or present substanc

In [139]:
claim = selected_claims[2]["claim"]

print("CLAIM:")
print(claim)

context = retrieve_evidence(claim)

print("\nRETRIEVED EVIDENCE:")
print(context[:3000])

result = classify_nli(
    context[:1500],
    claim
)

print("\nNLI RESULT:")
print(result)

print("\nFINAL LABEL:")
print(map_label(result["label"]))

CLAIM:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

RETRIEVED EVIDENCE:
Effect of homocysteine lowering on mortality and vascular disease in advanced chronic kidney disease and end-stage renal disease: a randomized controlled trial. CONTEXT High plasma homocysteine levels are a risk factor for mortality and vascular disease in observational studies of patients with chronic kidney disease. Folic acid and B vitamins decrease homocysteine levels in this population but whether they lower mortality is unknown.   
 OBJECTIVE To determine whether high doses of folic acid and B vitamins administered daily reduce mortality in patients with chronic kidney disease.   
 DESIGN, SETTING, AND PARTICIPANTS Double-blind randomized controlled trial (2001-2006) in 36 US Department of Veterans Affairs medical centers. Median follow-up was 3.2 years for 2056 participants aged 21 years or older with advanced chronic kidney diseas

In [140]:
import re

claim = selected_claims[0]["claim"]

context = retrieve_evidence(claim)

sentences = re.split(r'(?<=[.!?])\s+', context)

print("NUMBER OF SENTENCES:", len(sentences))

for i, sent in enumerate(sentences[:20]):
    print("\n", i)
    print(sent)

NUMBER OF SENTENCES: 24

 0
Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.

 1
DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.

 2
SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospitals in regions with lower levels of participation in that survey.

 3
SAMPLE 32,441 archived appendix samples fixed in formalin and embedded in paraffin and tested for the pr

In [141]:
best_sentence = sentences[4]

print(best_sentence)

result = classify_nli(
    best_sentence,
    claim
)

print(result)

print(map_label(result["label"]))

RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).
{'label': 'neutral', 'scores': [0.00966644287109375, 0.98291015625, 0.007568359375]}
SCOPE-MISMATCH


In [142]:
print(classify_nli(
    "Policies requiring discontinuation of methadone in 32% of all programs.",
    "32% of liver transplantation programs required patients to discontinue methadone treatment."
))

print(classify_nli(
    "Cats are animals.",
    "Cats are animals."
))

print(classify_nli(
    "Cats are animals.",
    "Cats are plants."
))

{'label': 'neutral', 'scores': [0.0003921985626220703, 0.99853515625, 0.0012950897216796875]}
{'label': 'entailment', 'scores': [0.998046875, 0.0017747879028320312, 0.00034880638122558594]}
{'label': 'contradiction', 'scores': [0.00013077259063720703, 0.0003540515899658203, 0.99951171875]}


In [143]:
def mistral_verify(claim, evidence):

    prompt = f"""
[INST]

Claim:
{claim}

Evidence:
{evidence}

Determine whether the claim is:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Answer with ONLY one word:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

[/INST]
"""

    inputs = mistral_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

    response = mistral_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [144]:
claim = selected_claims[0]["claim"]

context = retrieve_evidence(claim)

result = mistral_verify(
    claim,
    context[:1500]
)

print(result)


[INST]

Claim:
1 in 5 million in UK have abnormal PrP positivity.

Evidence:
Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   
 DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   
 SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospitals in regions with lower levels of participation in that survey.    SAMPLE 32,441 archived appendix samples fixed in formal

In [145]:
def mistral_verify(claim, evidence):

    prompt = f"""[INST]
Claim: {claim}

Evidence: {evidence}

Choose exactly one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Output only the label.
[/INST]"""

    inputs = mistral_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

    response = mistral_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    response = response.split("[/INST]")[-1].strip().upper()

    if "CONTRADICTED" in response:
        return "CONTRADICTED"
    elif "PARTIALLY_SUPPORTED" in response:
        return "PARTIALLY_SUPPORTED"
    elif "SUPPORTED" in response:
        return "SUPPORTED"
    else:
        return response

In [146]:
for idx, item in enumerate(selected_claims):

    claim = item["claim"]

    context = retrieve_evidence(claim)

    verdict = mistral_verify(
        claim,
        context[:1500]
    )

    print("\n" + "="*80)
    print("CLAIM:")
    print(claim)

    print("\nMISTRAL:")
    print(verdict)


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

MISTRAL:
SUPPORTED

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

MISTRAL:
SUPPORTED

CLAIM:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

MISTRAL:
SUPPORTED


In [147]:
test_claims = []

for i in range(20):

    item = claims_dataset["test"][i]

    test_claims.append({
        "claim": item["claim"],
        "true_label": item["evidence_label"]
    })

print("Number of claims:", len(test_claims))

Number of claims: 20


In [148]:
for x in test_claims[:5]:
    print("\nCLAIM:")
    print(x["claim"])

    print("TRUE:")
    print(x["true_label"])


CLAIM:
10-20% of people with severe mental disorder receive no treatment in low and middle income countries.
TRUE:


CLAIM:
25% of patients with melanoma and an objective response to PD-1 blockade will experience a progression in their melanoma.
TRUE:


CLAIM:
50% of patients exposed to radiation have activated markers of myofibroblasts.
TRUE:


CLAIM:
8% of burn patients are admitted for hospitalization and further treatment after appearing at hospital emergency wards or outpatient clinics.
TRUE:


CLAIM:
A breast cancer patient's capacity to metabolize tamoxifen has no effect on treatment outcome.
TRUE:



In [149]:
for i in range(5):

    item = claims_dataset["test"][i]

    print("\nCLAIM:")
    print(item["claim"])

    print("LABEL:")
    print(item["evidence_label"])


CLAIM:
10-20% of people with severe mental disorder receive no treatment in low and middle income countries.
LABEL:


CLAIM:
25% of patients with melanoma and an objective response to PD-1 blockade will experience a progression in their melanoma.
LABEL:


CLAIM:
50% of patients exposed to radiation have activated markers of myofibroblasts.
LABEL:


CLAIM:
8% of burn patients are admitted for hospitalization and further treatment after appearing at hospital emergency wards or outpatient clinics.
LABEL:


CLAIM:
A breast cancer patient's capacity to metabolize tamoxifen has no effect on treatment outcome.
LABEL:



In [150]:
print(claims_dataset["validation"][0])

{'id': 1, 'claim': '0-dimensional biomaterials show inductive properties.', 'evidence_doc_id': '', 'evidence_label': '', 'evidence_sentences': [], 'cited_doc_ids': [31715818]}


In [151]:
print(claims_dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'],
        num_rows: 1261
    })
    test: Dataset({
        features: ['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'],
        num_rows: 300
    })
    validation: Dataset({
        features: ['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'],
        num_rows: 450
    })
})


In [152]:
print(claims_dataset["train"][0])

{'id': 0, 'claim': '0-dimensional biomaterials lack inductive properties.', 'evidence_doc_id': '', 'evidence_label': '', 'evidence_sentences': [], 'cited_doc_ids': [31715818]}


In [153]:
print(corpus_dataset["train"][0])

{'doc_id': 4983, 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'abstract': ['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.', 'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).', 'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.', 'In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms.', 'In the posterior limb of the int

In [154]:
print(corpus_dataset)

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'title', 'abstract', 'structured'],
        num_rows: 5183
    })
})


In [155]:
print(corpus_dataset["train"].column_names)

['doc_id', 'title', 'abstract', 'structured']


In [156]:
print(claims_dataset["train"][0:5])

{'id': [0, 2, 4, 6, 9], 'claim': ['0-dimensional biomaterials lack inductive properties.', '1 in 5 million in UK have abnormal PrP positivity.', '1-1% of colorectal cancer patients are diagnosed with regional or distant metastases.', '10% of sudden infant death syndrome (SIDS) deaths happen in newborns aged less than 6 months.', '32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.'], 'evidence_doc_id': ['', '13734012', '', '', '44265107'], 'evidence_label': ['', 'CONTRADICT', '', '', 'SUPPORT'], 'evidence_sentences': [[], [4], [], [], [15]], 'cited_doc_ids': [[31715818], [13734012], [22942787], [2613775], [44265107]]}


In [157]:
supported = 0
contradict = 0
unlabeled = 0

for item in claims_dataset["train"]:

    label = item["evidence_label"]

    if label == "SUPPORT":
        supported += 1

    elif label == "CONTRADICT":
        contradict += 1

    else:
        unlabeled += 1

print("SUPPORT:", supported)
print("CONTRADICT:", contradict)
print("UNLABELED:", unlabeled)

SUPPORT: 616
CONTRADICT: 341
UNLABELED: 304


In [158]:
labeled_claims = []

for item in claims_dataset["train"]:

    if item["evidence_label"] in ["SUPPORT", "CONTRADICT"]:

        labeled_claims.append(item)

print("Labeled claims:", len(labeled_claims))

print("\nExample:")
print(labeled_claims[0]["claim"])
print("Label:", labeled_claims[0]["evidence_label"])

Labeled claims: 957

Example:
1 in 5 million in UK have abnormal PrP positivity.
Label: CONTRADICT


In [159]:
evaluation_claims = labeled_claims[:20]

print("Claims selected:", len(evaluation_claims))

Claims selected: 20


In [160]:
for i, item in enumerate(evaluation_claims):

    print("\n" + "="*80)
    print("CLAIM", i+1)

    print("\nClaim:")
    print(item["claim"])

    print("\nGold Label:")
    print(item["evidence_label"])


CLAIM 1

Claim:
1 in 5 million in UK have abnormal PrP positivity.

Gold Label:
CONTRADICT

CLAIM 2

Claim:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

Gold Label:
SUPPORT

CLAIM 3

Claim:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

Gold Label:
SUPPORT

CLAIM 4

Claim:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

Gold Label:
SUPPORT

CLAIM 5

Claim:
76-85% of people with severe mental disorder receive no treatment in low and middle income countries.

Gold Label:
SUPPORT

CLAIM 6

Claim:
A T helper 2 cell (Th2) environment impedes disease development in patients with systemic lupus erythematosus (SLE).

Gold Label:
CONTRADICT

CLAIM 7

Claim:
A T helper 2 cell (Th2) environment impedes disease development in patients with systemic lupus erythematosus (SLE).

Gold Label:
CONTR

In [161]:
unique_claims = {}

for item in labeled_claims:

    claim = item["claim"]

    if claim not in unique_claims:
        unique_claims[claim] = item

unique_claims = list(unique_claims.values())

print("Original labeled claims:", len(labeled_claims))
print("Unique labeled claims:", len(unique_claims))

Original labeled claims: 957
Unique labeled claims: 504


In [162]:
evaluation_claims = unique_claims[:20]

print("Evaluation claims:", len(evaluation_claims))

Evaluation claims: 20


In [163]:
deberta_results = []

for item in evaluation_claims:

    claim = item["claim"]
    gold_label = item["evidence_label"]

    evidence = retrieve_evidence(claim)

    result = classify_nli(
        evidence[:1500],
        claim
    )

    predicted = result["label"]

    deberta_results.append({
        "claim": claim,
        "gold": gold_label,
        "predicted": predicted
    })

    print("\n" + "="*80)
    print("CLAIM:")
    print(claim)

    print("\nGOLD:")
    print(gold_label)

    print("\nPREDICTED:")
    print(predicted)


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

GOLD:
CONTRADICT

PREDICTED:
neutral

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

GOLD:
SUPPORT

PREDICTED:
neutral

CLAIM:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

GOLD:
SUPPORT

PREDICTED:
neutral

CLAIM:
76-85% of people with severe mental disorder receive no treatment in low and middle income countries.

GOLD:
SUPPORT

PREDICTED:
neutral

CLAIM:
A T helper 2 cell (Th2) environment impedes disease development in patients with systemic lupus erythematosus (SLE).

GOLD:
CONTRADICT

PREDICTED:
contradiction

CLAIM:
A breast cancer patient's capacity to metabolize tamoxifen influences treatment outcome.

GOLD:
SUPPORT

PREDICTED:
entailment

CLAIM:
A country's Vaccine Alliance (GAVI) eligibility is not indictivate of accelerated adoption of the Hub vaccine.

GOLD:
CONTRADICT

PREDICTE

In [164]:
claim = evaluation_claims[0]

print("CLAIM:")
print(claim["claim"])

print("\nDOC ID:")
print(claim["evidence_doc_id"])

print("\nEVIDENCE SENTENCE IDS:")
print(claim["evidence_sentences"])

CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

DOC ID:
13734012

EVIDENCE SENTENCE IDS:
[4]


In [165]:
gold_doc_id = int(claim["evidence_doc_id"])

gold_doc = None

for doc in corpus_dataset["train"]:

    if doc["doc_id"] == gold_doc_id:
        gold_doc = doc
        break

print("TITLE:")
print(gold_doc["title"])

print("\nABSTRACT SENTENCES:")

for i, sent in enumerate(gold_doc["abstract"]):
    print(f"\n{i}:")
    print(sent)

TITLE:
Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey

ABSTRACT SENTENCES:

0:
OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   


1:
DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   


2:
SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospitals in regions with lower levels of participation in that survey.   

3:
SAMPLE 32,441 archived appendix samples fixed in formalin and embedded in paraffin and te

In [166]:
gold_sentence = gold_doc["abstract"][4]

print(gold_sentence)

print("\nNLI RESULT:")

print(
    classify_nli(
        gold_sentence,
        claim["claim"]
    )
)

RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).

NLI RESULT:
{'label': 'neutral', 'scores': [0.00966644287109375, 0.98291015625, 0.007568359375]}


In [167]:
def convert_prediction(label):

    label = label.lower()

    if label == "entailment":
        return "SUPPORT"

    elif label == "contradiction":
        return "CONTRADICT"

    else:
        return "NEI"

In [168]:
correct = 0
total = len(evaluation_claims)

for claim_data in evaluation_claims:

    claim = claim_data["claim"]

    gold = claim_data["evidence_label"]

    context = retrieve_evidence(claim)

    result = classify_nli(
        context[:1500],
        claim
    )

    pred = convert_prediction(result["label"])

    if pred == gold:
        correct += 1

accuracy = correct / total

print("Correct:", correct)
print("Total:", total)
print("Accuracy:", round(accuracy * 100, 2), "%")

Correct: 10
Total: 20
Accuracy: 50.0 %


In [169]:
from collections import Counter

confusion = Counter()

for claim_data in evaluation_claims:

    claim = claim_data["claim"]

    gold = claim_data["evidence_label"]

    context = retrieve_evidence(claim)

    result = classify_nli(
        context[:1500],
        claim
    )

    pred = convert_prediction(result["label"])

    confusion[(gold, pred)] += 1

print("CONFUSION MATRIX\n")

for k, v in sorted(confusion.items()):
    print(k, ":", v)

CONFUSION MATRIX

('CONTRADICT', 'CONTRADICT') : 3
('CONTRADICT', 'NEI') : 2
('CONTRADICT', 'SUPPORT') : 1
('SUPPORT', 'NEI') : 7
('SUPPORT', 'SUPPORT') : 7


In [170]:
for name in list(globals().keys()):
    if "mistral" in name.lower():
        print(name)

mistral_tokenizer
mistral_model
mistral_prompt
mistral_answer
run_mistral
mistral_verify


In [171]:
for name in list(globals().keys()):
    if "generate" in name.lower():
        print(name)

generated_summary


In [172]:
print(run_mistral)

<function run_mistral at 0x7c9ef1d3d620>


In [173]:
print(mistral_verify)

<function mistral_verify at 0x7c9ef1d44e00>


In [174]:
import inspect

print(inspect.getsource(run_mistral))

def run_mistral(claim, context):

    prompt = f"""
Claim:
{claim}

Evidence:
{context}

Determine whether the claim is supported.

Explain using the evidence.
"""

    messages = f"<s>[INST] {prompt} [/INST]"

    inputs = mistral_tokenizer(
        messages,
        return_tensors="pt"
    ).to("cuda")

    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=mistral_tokenizer.eos_token_id
    )

    answer = mistral_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer



In [175]:
print(inspect.getsource(mistral_verify))

def mistral_verify(claim, evidence):

    prompt = f"""[INST]
Claim: {claim}

Evidence: {evidence}

Choose exactly one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Output only the label.
[/INST]"""

    inputs = mistral_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

    response = mistral_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    response = response.split("[/INST]")[-1].strip().upper()

    if "CONTRADICTED" in response:
        return "CONTRADICTED"
    elif "PARTIALLY_SUPPORTED" in response:
        return "PARTIALLY_SUPPORTED"
    elif "SUPPORTED" in response:
        return "SUPPORTED"
    else:
        return response



In [176]:
def normalize_mistral_label(label):

    if label == "SUPPORTED":
        return "SUPPORT"

    elif label == "CONTRADICTED":
        return "CONTRADICT"

    else:
        return "NEI"

In [177]:
correct = 0
total = min(50, len(unique_claims))

for i in range(total):

    claim = unique_claims[i]

    evidence = retrieve_evidence(
        claim["claim"]
    )

    prediction = normalize_mistral_label(mistral_verify(
        claim["claim"],
        evidence[:1500]
    ))

    gold = claim["evidence_label"]

    if prediction == gold:
        correct += 1

    print("\n" + "="*60)
    print("CLAIM:", claim["claim"])
    print("GOLD:", gold)
    print("PREDICTED:", prediction)

print("\nAccuracy:",
      correct / total * 100)


CLAIM: 1 in 5 million in UK have abnormal PrP positivity.
GOLD: CONTRADICT
PREDICTED: SUPPORT

CLAIM: 32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
GOLD: SUPPORT
PREDICTED: SUPPORT

CLAIM: 40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.
GOLD: SUPPORT
PREDICTED: SUPPORT

CLAIM: 76-85% of people with severe mental disorder receive no treatment in low and middle income countries.
GOLD: SUPPORT
PREDICTED: SUPPORT

CLAIM: A T helper 2 cell (Th2) environment impedes disease development in patients with systemic lupus erythematosus (SLE).
GOLD: CONTRADICT
PREDICTED: CONTRADICT

CLAIM: A breast cancer patient's capacity to metabolize tamoxifen influences treatment outcome.
GOLD: SUPPORT
PREDICTED: SUPPORT

CLAIM: A country's Vaccine Alliance (GAVI) eligibility is not indictivate of accelerated adoption of the Hub vaccine.
GOLD: CONTRADICT
PREDICTED: SUPPORT

CLAIM: A

Testing smtg


In [178]:
print(type(nli_classifier))

<class 'transformers.pipelines.text_classification.TextClassificationPipeline'>


In [179]:
claim = unique_claims[0]["claim"]

evidence = retrieve_evidence(claim)

result = nli_classifier(
    evidence,
    candidate_labels=[
        "supported",
        "contradicted",
        "not enough information"
    ]
)

print("CLAIM:")
print(claim)

print("\nRESULT:")
print(result)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1095 > 512). Running this sequence through the model will result in indexing errors


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

RESULT:
[{'label': 'entailment', 'score': 0.7403151988983154}]


In [180]:
claim = unique_claims[0]["claim"]

evidence = retrieve_evidence(claim)

evidence = evidence[:400]

result = nli_classifier(
    evidence,
    candidate_labels=[
        "supported",
        "contradicted",
        "not enough information"
    ]
)

print(result)

[{'label': 'not_entailment', 'score': 0.9697281122207642}]


Testing failed


mistral vs Qwen


In [181]:
import torch

print(torch.cuda.get_device_name(0))
print("Allocated:",
      round(torch.cuda.memory_allocated()/1024**3,2),"GB")
print("Reserved:",
      round(torch.cuda.memory_reserved()/1024**3,2),"GB")


Tesla T4
Allocated: 4.84 GB
Reserved: 6.07 GB


In [182]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

qwen_model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

qwen_tokenizer = AutoTokenizer.from_pretrained(
    qwen_model_name
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Qwen loaded!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen loaded!


In [183]:
def qwen_verify(claim, evidence):

    prompt = f"""
Claim: {claim}

Evidence: {evidence}

Choose exactly one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Output only the label.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

    response = qwen_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    response = response.upper()

    if "CONTRADICTED" in response:
        return "CONTRADICTED"

    elif "PARTIALLY_SUPPORTED" in response:
        return "PARTIALLY_SUPPORTED"

    elif "SUPPORTED" in response:
        return "SUPPORTED"

    else:
        return response

In [184]:
claim = unique_claims[0]["claim"]

evidence = retrieve_evidence(claim)

prediction = qwen_verify(
    claim,
    evidence[:1500]
)

print("CLAIM:")
print(claim)

print("\nPREDICTION:")
print(prediction)

CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

PREDICTION:
CONTRADICTED


In [185]:
def normalize_qwen_label(label):

    if label == "SUPPORTED":
        return "SUPPORT"

    elif label == "CONTRADICTED":
        return "CONTRADICT"

    else:
        return "NEI"

In [186]:
correct = 0
total = 10

for i in range(total):

    claim = unique_claims[i]

    evidence = retrieve_evidence(
        claim["claim"]
    )

    prediction = normalize_qwen_label(
        qwen_verify(
            claim["claim"],
            evidence[:1500]
        )
    )

    gold = claim["evidence_label"]

    if prediction == gold:
        correct += 1

    print("\n" + "="*60)
    print("CLAIM:", claim["claim"])
    print("GOLD:", gold)
    print("PREDICTED:", prediction)

print("\nCorrect:", correct)
print("Total:", total)
print("Accuracy:", round(correct / total * 100, 2), "%")


CLAIM: 1 in 5 million in UK have abnormal PrP positivity.
GOLD: CONTRADICT
PREDICTED: CONTRADICT

CLAIM: 32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
GOLD: SUPPORT
PREDICTED: CONTRADICT

CLAIM: 40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.
GOLD: SUPPORT
PREDICTED: CONTRADICT

CLAIM: 76-85% of people with severe mental disorder receive no treatment in low and middle income countries.
GOLD: SUPPORT
PREDICTED: CONTRADICT

CLAIM: A T helper 2 cell (Th2) environment impedes disease development in patients with systemic lupus erythematosus (SLE).
GOLD: CONTRADICT
PREDICTED: CONTRADICT

CLAIM: A breast cancer patient's capacity to metabolize tamoxifen influences treatment outcome.
GOLD: SUPPORT
PREDICTED: CONTRADICT

CLAIM: A country's Vaccine Alliance (GAVI) eligibility is not indictivate of accelerated adoption of the Hub vaccine.
GOLD: CONTRADICT
PREDICTED: CO

In [187]:
claim = unique_claims[1]["claim"]

evidence = retrieve_evidence(claim)

prompt = f"""
Claim: {claim}

Evidence: {evidence[:1000]}

Choose exactly one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Output only the label.
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = qwen_tokenizer(
    text,
    return_tensors="pt"
).to("cuda")

outputs = qwen_model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

response = qwen_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Claim: 32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

Evidence: Liver transplantation and opioid dependence. ContextChronic hepatitis C is the leading cause for liver transplantation in the United States. Intravenous drug use, the major risk factor, accounts for approximately 60% of hepatitis C virus transmission. Information from the United Network of Organ Sharing (UNOS) does not address substance use among liver transplantation patients. ObjectiveTo identify addiction-related criteria for admission to the UNOS liver transplantation waiting list and posttransplantation problems experienced by patients who are prescribed maintenance methadone. Design, Setting, and ParticipantsMail survey of all 97 adult US liver transplantation programs (belonging to UNOS) in March 2000 with telephone follow-up conducted in May and June 2000.Main Outcome MeasuresProgr

In [188]:
claim = unique_claims[1]["claim"]

evidence = retrieve_evidence(claim)

print(len(evidence))

print("\n====================\n")

print(evidence)

4253


Liver transplantation and opioid dependence. ContextChronic hepatitis C is the leading cause for liver transplantation in the United States. Intravenous drug use, the major risk factor, accounts for approximately 60% of hepatitis C virus transmission. Information from the United Network of Organ Sharing (UNOS) does not address substance use among liver transplantation patients. ObjectiveTo identify addiction-related criteria for admission to the UNOS liver transplantation waiting list and posttransplantation problems experienced by patients who are prescribed maintenance methadone. Design, Setting, and ParticipantsMail survey of all 97 adult US liver transplantation programs (belonging to UNOS) in March 2000 with telephone follow-up conducted in May and June 2000.Main Outcome MeasuresPrograms' acceptance and management of patients with past or present substance use disorder. ResultsOf the 97 programs surveyed, 87 (90%) responded. All accept applicants with a history of alcoholis

In [189]:
corpus_lookup = {}

for doc in corpus_dataset["train"]:
    corpus_lookup[doc["doc_id"]] = doc

print("Loaded", len(corpus_lookup), "documents")

Loaded 5183 documents


In [190]:
claim = evaluation_claims[1]

doc_id = int(claim["evidence_doc_id"])

sentence_id = claim["evidence_sentences"][0]

doc = corpus_lookup[doc_id]

evidence = doc["abstract"][sentence_id]

print(evidence)

Policies requiring discontinuation of methadone in 32% of all programs contradict the evidence base for efficacy of long-term replacement therapies and potentially result in relapse of previously stable patients.


In [191]:
qwen_verify(claim["claim"], evidence)

'CONTRADICTED'

In [192]:
mistral_verify(claim["claim"], evidence)

'CONTRADICTED'

In [193]:
def qwen_verify_v2(claim, evidence):

    prompt = f"""
You are a scientific fact checker.

Claim:
{claim}

Evidence:
{evidence}

Determine the relationship between the claim and evidence.

Definitions:

SUPPORT:
The evidence directly supports the claim.

CONTRADICT:
The evidence directly contradicts the claim.

NEI:
The evidence does not provide enough information.

Answer with ONLY one word:

SUPPORT
CONTRADICT
NEI
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

    response = qwen_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    response = response.upper()

    if "CONTRADICT" in response:
        return "CONTRADICT"

    elif "SUPPORT" in response:
        return "SUPPORT"

    elif "NEI" in response:
        return "NEI"

    else:
        return response

In [194]:
claim = evaluation_claims[1]

doc_id = int(claim["evidence_doc_id"])

sentence_id = claim["evidence_sentences"][0]

doc = corpus_lookup[doc_id]

evidence = doc["abstract"][sentence_id]

print("CLAIM:")
print(claim["claim"])

print("\nEVIDENCE:")
print(evidence)

print("\nQWEN V2:")
print(
    qwen_verify_v2(
        claim["claim"],
        evidence
    )
)

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

EVIDENCE:
Policies requiring discontinuation of methadone in 32% of all programs contradict the evidence base for efficacy of long-term replacement therapies and potentially result in relapse of previously stable patients.

QWEN V2:
CONTRADICT


both models not working, now trying to find root cause

In [195]:
corpus_lookup = {}

for doc in corpus_dataset["train"]:
    corpus_lookup[doc["doc_id"]] = doc

print("Loaded", len(corpus_lookup), "documents")

Loaded 5183 documents


In [196]:
evaluation_gold = []

for item in unique_claims:

    if (
        item["evidence_doc_id"] != ""
        and len(item["evidence_sentences"]) > 0
    ):

        evaluation_gold.append(item)

print("Total gold-evidence claims:",
      len(evaluation_gold))

Total gold-evidence claims: 504


In [197]:
claim = evaluation_gold[0]

doc_id = int(claim["evidence_doc_id"])

sentence_id = claim["evidence_sentences"][0]

doc = corpus_lookup[doc_id]

evidence = doc["abstract"][sentence_id]

print("CLAIM:")
print(claim["claim"])

print("\nGOLD:")
print(claim["evidence_label"])

print("\nEVIDENCE:")
print(evidence)

CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

GOLD:
CONTRADICT

EVIDENCE:
RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).


In [198]:
def normalize_mistral(label):

    if label == "SUPPORTED":
        return "SUPPORT"

    elif label == "CONTRADICTED":
        return "CONTRADICT"

    else:
        return "NEI"

In [199]:
correct = 0
total = 20

for i in range(total):

    claim = evaluation_gold[i]

    doc_id = int(claim["evidence_doc_id"])

    sentence_id = claim["evidence_sentences"][0]

    evidence = corpus_lookup[doc_id]["abstract"][sentence_id]

    prediction = normalize_mistral(
        mistral_verify(
            claim["claim"],
            evidence
        )
    )

    gold = claim["evidence_label"]

    if prediction == gold:
        correct += 1

    print("\n" + "="*60)
    print("CLAIM:", claim["claim"])
    print("GOLD:", gold)
    print("PREDICTED:", prediction)

print("\nCorrect:", correct)
print("Total:", total)
print("Accuracy:",
      round(correct/total*100,2),
      "%")


CLAIM: 1 in 5 million in UK have abnormal PrP positivity.
GOLD: CONTRADICT
PREDICTED: SUPPORT

CLAIM: 32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
GOLD: SUPPORT
PREDICTED: CONTRADICT

CLAIM: 40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.
GOLD: SUPPORT
PREDICTED: SUPPORT

CLAIM: 76-85% of people with severe mental disorder receive no treatment in low and middle income countries.
GOLD: SUPPORT
PREDICTED: NEI

CLAIM: A T helper 2 cell (Th2) environment impedes disease development in patients with systemic lupus erythematosus (SLE).
GOLD: CONTRADICT
PREDICTED: NEI

CLAIM: A breast cancer patient's capacity to metabolize tamoxifen influences treatment outcome.
GOLD: SUPPORT
PREDICTED: SUPPORT

CLAIM: A country's Vaccine Alliance (GAVI) eligibility is not indictivate of accelerated adoption of the Hub vaccine.
GOLD: CONTRADICT
PREDICTED: CONTRADICT

CLAIM: A defi

In [200]:
correct = 0
total = 20

for i in range(total):

    claim = evaluation_gold[i]

    doc_id = int(claim["evidence_doc_id"])

    sentence_id = claim["evidence_sentences"][0]

    evidence = corpus_lookup[doc_id]["abstract"][sentence_id]

    prediction = qwen_verify_v2(
        claim["claim"],
        evidence
    )

    gold = claim["evidence_label"]

    if prediction == gold:
        correct += 1

    print("\n" + "="*60)
    print("CLAIM:", claim["claim"])
    print("GOLD:", gold)
    print("PREDICTED:", prediction)

print("\nCorrect:", correct)
print("Total:", total)
print("Accuracy:",
      round(correct/total*100,2),
      "%")


CLAIM: 1 in 5 million in UK have abnormal PrP positivity.
GOLD: CONTRADICT
PREDICTED: CONTRADICT

CLAIM: 32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
GOLD: SUPPORT
PREDICTED: CONTRADICT

CLAIM: 40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.
GOLD: SUPPORT
PREDICTED: CONTRADICT

CLAIM: 76-85% of people with severe mental disorder receive no treatment in low and middle income countries.
GOLD: SUPPORT
PREDICTED: CONTRADICT

CLAIM: A T helper 2 cell (Th2) environment impedes disease development in patients with systemic lupus erythematosus (SLE).
GOLD: CONTRADICT
PREDICTED: CONTRADICT

CLAIM: A breast cancer patient's capacity to metabolize tamoxifen influences treatment outcome.
GOLD: SUPPORT
PREDICTED: CONTRADICT

CLAIM: A country's Vaccine Alliance (GAVI) eligibility is not indictivate of accelerated adoption of the Hub vaccine.
GOLD: CONTRADICT
PREDICTED: CO

In [201]:
print(qwen_verify_v2(
    "Cats are animals.",
    "Cats are animals."
))

print(qwen_verify_v2(
    "Cats are animals.",
    "Cats are plants."
))

print(qwen_verify_v2(
    "The sky is blue.",
    "The sky has a color."
))

CONTRADICT
CONTRADICT
CONTRADICT


In [202]:
claim = "Cats are animals."
evidence = "Cats are animals."

prompt = f"""
You are a scientific fact checker.

Claim:
{claim}

Evidence:
{evidence}

Determine the relationship between the claim and evidence.

Definitions:

SUPPORT:
The evidence directly supports the claim.

CONTRADICT:
The evidence directly contradicts the claim.

NEI:
The evidence does not provide enough information.

Answer with ONLY one word:

SUPPORT
CONTRADICT
NEI
"""

messages = [{"role": "user", "content": prompt}]

text = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = qwen_tokenizer(
    text,
    return_tensors="pt"
).to("cuda")

outputs = qwen_model.generate(
    **inputs,
    max_new_tokens=20,
    do_sample=False
)

response = qwen_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

You are a scientific fact checker.

Claim:
Cats are animals.

Evidence:
Cats are animals.

Determine the relationship between the claim and evidence.

Definitions:

SUPPORT:
The evidence directly supports the claim.

CONTRADICT:
The evidence directly contradicts the claim.

NEI:
The evidence does not provide enough information.

Answer with ONLY one word:

SUPPORT
CONTRADICT
NEI

assistant
SUPPORT


In [203]:
print(qwen_verify_v2(
    "Cats are animals.",
    "Cats are plants."
))

print(qwen_verify_v2(
    "The sky is blue.",
    "The sky has a color."
))

CONTRADICT
CONTRADICT


so mistral is only better thn qwen


In [204]:
def qwen_extract_evidence(claim, document):

    prompt = f"""
Claim:
{claim}

Document:
{document}

Extract the ONE sentence that is most useful
for verifying the claim.

Output only the sentence.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    response = qwen_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [205]:
claim = evaluation_gold[1]

doc_id = int(claim["evidence_doc_id"])

document = " ".join(
    corpus_lookup[doc_id]["abstract"]
)

print(
    qwen_extract_evidence(
        claim["claim"],
        document
    )
)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Claim:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

Document:
ContextChronic hepatitis C is the leading cause for liver transplantation in the United States. Intravenous drug use, the major risk factor, accounts for approximately 60% of hepatitis C virus transmission. Information from the United Network of Organ Sharing (UNOS) does not address substance use among liver transplantation patients. ObjectiveTo identify addiction-related criteria for admission to the UNOS liver transplantation waiting list and posttransplantation problems experienced by patients who are prescribed maintenance methadone. Design, Setting, and ParticipantsMail survey of all 97 adult US liver transplantation programs (belonging to UNOS) in March 2000 with telephone follow-up conducted in May and June 2000.Main Outcome MeasuresPrograms' acceptance and management of patients wi

In [206]:
def get_gold_sentence(claim):

    doc_id = int(claim["evidence_doc_id"])

    sent_id = claim["evidence_sentences"][0]

    return corpus_lookup[doc_id]["abstract"][sent_id]

In [207]:
for i in range(10):

    claim = evaluation_gold[i]

    doc_id = int(claim["evidence_doc_id"])

    document = " ".join(
        corpus_lookup[doc_id]["abstract"]
    )

    extracted = qwen_extract_evidence(
        claim["claim"],
        document
    )

    gold = get_gold_sentence(claim)

    print("\n" + "="*80)

    print("CLAIM:")
    print(claim["claim"])

    print("\nGOLD:")
    print(gold)

    print("\nQWEN:")
    print(extracted)


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

GOLD:
RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).

QWEN:
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Claim:
1 in 5 million in UK have abnormal PrP positivity.

Document:
OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   
 DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   
 SETTING Archived appendix samples from the pathology de

In [208]:
def qwen_extract_evidence_clean(claim, document):

    prompt = f"""
Claim:
{claim}

Document:
{document}

Extract the ONE sentence that is most useful for verifying the claim.

Output only the sentence.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    response = qwen_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if "assistant" in response:
        response = response.split("assistant")[-1]

    return response.strip()

In [209]:
def get_gold_sentence(claim):

    doc_id = int(claim["evidence_doc_id"])

    sent_id = claim["evidence_sentences"][0]

    return corpus_lookup[doc_id]["abstract"][sent_id]

In [210]:
results = []

for i in range(20):

    claim = evaluation_gold[i]

    doc_id = int(claim["evidence_doc_id"])

    document = " ".join(
        corpus_lookup[doc_id]["abstract"]
    )

    gold = get_gold_sentence(claim)

    extracted = qwen_extract_evidence_clean(
        claim["claim"],
        document
    )

    results.append({
        "claim": claim["claim"],
        "gold": gold,
        "extracted": extracted
    })

    print("\n" + "="*80)

    print("CLAIM:")
    print(claim["claim"])

    print("\nGOLD:")
    print(gold)

    print("\nEXTRACTED:")
    print(extracted)


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

GOLD:
RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).

EXTRACTED:
Of the 32,441 appendix samples, 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

GOLD:
Policies requiring discontinuation of methadone in 32% of all programs contradict the evidence base for efficacy of long-term replacement therapies and potentially result in relapse of previously stable patients.

EXTRACTED:
Policies requiring discontinuation of methadone in 32% of all programs contradict the evidence base for efficacy of long-term replacement therapies and potentially result in relapse of previously stable patients.

C

In [211]:
import pandas as pd

df = pd.DataFrame(results)

df.to_csv(
    "qwen_evidence_results.csv",
    index=False
)

print(df.head())

                                               claim  \
0  1 in 5 million in UK have abnormal PrP positiv...   
1  32% of liver transplantation programs required...   
2  40mg/day dosage of folic acid and 2mg/day dosa...   
3  76-85% of people with severe mental disorder r...   
4  A T helper 2 cell (Th2) environment impedes di...   

                                                gold  \
0  RESULTS Of the 32,441 appendix samples 16 were...   
1  Policies requiring discontinuation of methadon...   
2  RESULTS Mean baseline homocysteine level was 2...   
3  Although disorder severity was correlated with...   
4  We report that activation of basophils by auto...   

                                           extracted  
0  Of the 32,441 appendix samples, 16 were positi...  
1  Policies requiring discontinuation of methadon...  
2  Treatment with high doses of folic acid and B ...  
3  Between 33.1% (Colombia) and 80.9% (Nigeria) o...  
4  Activation of basophils by autoreactive IgE ca..

In [212]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

embed_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

scores = []

for row in results:

    gold = row["gold"]
    extracted = row["extracted"]

    emb = embed_model.encode(
        [gold, extracted]
    )

    score = cosine_similarity(
        [emb[0]],
        [emb[1]]
    )[0][0]

    scores.append(score)

print("Average Similarity:",
      round(np.mean(scores), 3))

print("Min Similarity:",
      round(np.min(scores), 3))

print("Max Similarity:",
      round(np.max(scores), 3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Average Similarity: 0.67
Min Similarity: 0.188
Max Similarity: 1.0


In [213]:
for i, score in enumerate(scores):

    if score > 0.8:
        label = "EXCELLENT"

    elif score > 0.6:
        label = "GOOD"

    else:
        label = "BAD"

    print(i,
          round(score,3),
          label)

0 0.989 EXCELLENT
1 1.0 EXCELLENT
2 0.456 BAD
3 0.715 GOOD
4 0.994 EXCELLENT
5 0.188 BAD
6 0.38 BAD
7 0.78 GOOD
8 0.561 BAD
9 0.561 BAD
10 0.759 GOOD
11 0.683 GOOD
12 0.548 BAD
13 0.4 BAD
14 0.996 EXCELLENT
15 0.44 BAD
16 0.785 GOOD
17 0.647 GOOD
18 0.647 GOOD
19 0.864 EXCELLENT


building pipeline
Claim -> Retriever -> Qwen -> Mistral

In [214]:
def fact_check_claim(claim_text):

    # Step 1: Retrieve document
    document = retrieve_evidence(claim_text)

    # Step 2: Extract evidence sentence using Qwen
    evidence = qwen_extract_evidence_clean(
        claim_text,
        document
    )

    # Step 3: Verify using Mistral
    verdict = mistral_verify(
        claim_text,
        evidence
    )

    return {
        "claim": claim_text,
        "evidence": evidence,
        "verdict": verdict
    }

In [215]:
claim = unique_claims[0]["claim"]

result = fact_check_claim(claim)

print("CLAIM:")
print(result["claim"])

print("\nEVIDENCE:")
print(result["evidence"])

print("\nVERDICT:")
print(result["verdict"])

CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

EVIDENCE:
Of the 32,441 appendix samples, 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).

VERDICT:
THE CLAIM "1 IN 5 MILLION IN UK


mistral is returning an explanation instead of just a label, trying to fix the verifier output

In [216]:
claim = "1 in 5 million in UK have abnormal PrP positivity."

evidence = """
Of the 32,441 appendix samples, 16 were positive for abnormal PrP,
indicating an overall prevalence of 493 per million population.
"""

result = mistral_verify(
    claim,
    evidence
)

print(result)

PARTIALLY_SUPPORTED


In [217]:
claim = unique_claims[0]["claim"]

document = retrieve_evidence(claim)

print("="*60)
print("DOCUMENT:")
print(document[:1000])

evidence = qwen_extract_evidence_clean(
    claim,
    document
)

print("\n" + "="*60)
print("EVIDENCE:")
print(evidence)

verdict = mistral_verify(
    claim,
    evidence
)

print("\n" + "="*60)
print("VERDICT:")
print(verdict)

DOCUMENT:
Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   
 DESIGN Irreversibly unlinked and anonymised large scale survey of archived appendix samples.   
 SETTING Archived appendix samples from the pathology departments of 41 UK hospitals participating in the earlier survey, and additional hospitals in regions with lower levels of participation in that survey.    SAMPLE 32,441 archived appendix samples fixed in formalin and embedded in paraffin and tested for the presence of abnormal 

In [218]:
print(mistral_verify)

<function mistral_verify at 0x7c9ef1d44e00>


In [219]:
import inspect

print(inspect.getsource(mistral_verify))

def mistral_verify(claim, evidence):

    prompt = f"""[INST]
Claim: {claim}

Evidence: {evidence}

Choose exactly one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Output only the label.
[/INST]"""

    inputs = mistral_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

    response = mistral_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    response = response.split("[/INST]")[-1].strip().upper()

    if "CONTRADICTED" in response:
        return "CONTRADICTED"
    elif "PARTIALLY_SUPPORTED" in response:
        return "PARTIALLY_SUPPORTED"
    elif "SUPPORTED" in response:
        return "SUPPORTED"
    else:
        return response



In [220]:
claim = unique_claims[0]["claim"]

document = retrieve_evidence(claim)

evidence = qwen_extract_evidence_clean(
    claim,
    document
)

print(type(evidence))
print(repr(evidence))

<class 'str'>
'Of the 32,441 appendix samples, 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).'


In [221]:
verdict = mistral_verify(
    claim,
    evidence
)

print(repr(verdict))

'THE CLAIM "1 IN 5 MILLION IN UK'


In [222]:
claim = unique_claims[0]["claim"]

evidence = """
Of the 32,441 appendix samples, 16 were positive for abnormal PrP,
indicating an overall prevalence of 493 per million population.
"""

prompt = f"""[INST]
Claim: {claim}

Evidence: {evidence}

Choose exactly one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Output only the label.
[/INST]"""

inputs = mistral_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=2048
).to("cuda")

outputs = mistral_model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=False
)

raw_response = mistral_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(raw_response)

[INST]
Claim: 1 in 5 million in UK have abnormal PrP positivity.

Evidence: 
Of the 32,441 appendix samples, 16 were positive for abnormal PrP,
indicating an overall prevalence of 493 per million population.


Choose exactly one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Output only the label.
[/INST] PARTIALLY_SUPPORTED

Explanation: The claim states that 1 in 5 million in the UK have abnormal Pr


found the bug, replacing the mistral function

In [223]:
def mistral_verify(claim, evidence):

    prompt = f"""[INST]
Claim: {claim}

Evidence: {evidence}

Choose exactly one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Output only the label.
[/INST]"""

    inputs = mistral_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False
    )

    response = mistral_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).upper()

    # Search entire output for labels

    if "CONTRADICTED" in response:
        return "CONTRADICTED"

    if "PARTIALLY_SUPPORTED" in response:
        return "PARTIALLY_SUPPORTED"

    if "SUPPORTED" in response:
        return "SUPPORTED"

    return "UNKNOWN"

In [224]:
claim = unique_claims[0]["claim"]

result = fact_check_claim(claim)

print(result)

{'claim': '1 in 5 million in UK have abnormal PrP positivity.', 'evidence': 'Of the 32,441 appendix samples, 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).', 'verdict': 'CONTRADICTED'}


pipeline works! now end-to-end evaluation

In [225]:
correct = 0
total = 10

for i in range(total):

    claim = unique_claims[i]

    result = fact_check_claim(
        claim["claim"]
    )

    prediction = result["verdict"]

    gold = claim["evidence_label"]

    if prediction == gold:
        correct += 1

    print("\n" + "="*70)

    print("CLAIM:")
    print(claim["claim"])

    print("\nGOLD:")
    print(gold)

    print("\nPREDICTED:")
    print(prediction)

print("\n" + "="*70)
print("Correct:", correct)
print("Total:", total)
print("Accuracy:", correct / total * 100)


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

GOLD:
CONTRADICT

PREDICTED:
CONTRADICTED

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

GOLD:
SUPPORT

PREDICTED:
CONTRADICTED

CLAIM:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

GOLD:
SUPPORT

PREDICTED:
CONTRADICTED

CLAIM:
76-85% of people with severe mental disorder receive no treatment in low and middle income countries.

GOLD:
SUPPORT

PREDICTED:
CONTRADICTED

CLAIM:
A T helper 2 cell (Th2) environment impedes disease development in patients with systemic lupus erythematosus (SLE).

GOLD:
CONTRADICT

PREDICTED:
CONTRADICTED

CLAIM:
A breast cancer patient's capacity to metabolize tamoxifen influences treatment outcome.

GOLD:
SUPPORT

PREDICTED:
CONTRADICTED

CLAIM:
A country's Vaccine Alliance (GAVI) eligibility is not indictivate of accelerated adoption of the Hub vaccine.

GOLD:

In [226]:
def normalize_pipeline_label(label):

    if label == "CONTRADICTED":
        return "CONTRADICT"

    if label == "SUPPORTED":
        return "SUPPORT"

    return label

In [227]:
prediction = normalize_pipeline_label(
    result["verdict"]
)

mistral is predicting contradict for everything. figuring out the bug

In [228]:
claim = unique_claims[1]

result = fact_check_claim(
    claim["claim"]
)

print("CLAIM:")
print(claim["claim"])

print("\nGOLD:")
print(claim["evidence_label"])

print("\nEVIDENCE:")
print(result["evidence"])

print("\nVERDICT:")
print(result["verdict"])

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

GOLD:
SUPPORT

EVIDENCE:
Policies requiring discontinuation of methadone in 32% of all programs contradict the evidence base for efficacy of long-term replacement therapies and potentially result in relapse of previously stable patients.

VERDICT:
CONTRADICTED


In [229]:
claim = """
32% of liver transplantation programs required patients
to discontinue methadone treatment in 2001.
"""

evidence = """
Policies requiring discontinuation of methadone
in 32% of all programs.
"""

print(
    mistral_verify(
        claim,
        evidence
    )
)

CONTRADICTED


In [230]:
claim = "Cats are animals."

evidence = "Cats are animals."

print(
    mistral_verify(
        claim,
        evidence
    )
)

CONTRADICTED


In [231]:
claim = "Cats are plants."

evidence = "Cats are animals."

print(
    mistral_verify(
        claim,
        evidence
    )
)

CONTRADICTED


In [232]:
claim = "Cats are animals."
evidence = "Cats are animals."

prompt = f"""[INST]
Claim: {claim}

Evidence: {evidence}

Choose exactly one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Output only the label.
[/INST]"""

inputs = mistral_tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = mistral_model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=False
)

raw = mistral_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(raw)

[INST]
Claim: Cats are animals.

Evidence: Cats are animals.

Choose exactly one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED

Output only the label.
[/INST] SUPPORTED


figured out the bug, replacing the mistral function

In [233]:
import re

def parse_verdict(response):

    response = response.upper()

    if re.search(r"PARTIALLY[_ ]SUPPORTED", response):
        return "PARTIALLY_SUPPORTED"

    if re.search(r"PARTLY[_ ]SUPPORTED", response):
        return "PARTIALLY_SUPPORTED"

    if "PARTLY CONTRADICTED" in response:
        return "CONTRADICTED"

    if "CONTRADICTED" in response:
        return "CONTRADICTED"

    if "SUPPORTED" in response:
        return "SUPPORTED"

    return "PARTIALLY_SUPPORTED"


def mistral_verify(claim, evidence):

    prompt = f"""[INST]
You are a scientific fact checker.

Claim:
{claim}

Evidence:
{evidence}

Definitions:

SUPPORTED:
The evidence directly supports the claim.

CONTRADICTED:
The evidence directly contradicts the claim.

PARTIALLY_SUPPORTED:
The evidence is related to the claim but does not fully support or contradict it.

Important:
Do NOT decide based on individual words such as
"contradict", "increase", "decrease", or "not".

Reason about the meaning of the entire claim and the entire evidence.

Output ONLY one label:

SUPPORTED
CONTRADICTED
PARTIALLY_SUPPORTED
[/INST]"""

    inputs = mistral_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=mistral_tokenizer.eos_token_id
    )

    response = mistral_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Keep only generated portion
    response = response.split("[/INST]")[-1].strip()

    return parse_verdict(response)

In [234]:
claim = "32% of liver transplantation programs required patients to discontinue methadone treatment in 2001."

evidence = "Policies requiring discontinuation of methadone in 32% of all programs contradict the evidence base for efficacy of long-term replacement therapies and potentially result in relapse of previously stable patients."

print(
    mistral_verify(
        claim,
        evidence
    )
)

PARTIALLY_SUPPORTED


Experiment:
Qwen 2.5 + Mistral 7B

Observation:
Evidence extraction quality was generally good.
Verifier frequently produced PARTIALLY_SUPPORTED
for claims that were clearly SUPPORTED.
Verifier is weak.

Example:
Methadone discontinuation claim.

comparing Mistral against DeBERTa

In [235]:
from transformers import pipeline

nli_pipeline = pipeline(
    "text-classification",
    model="MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
    device=0
)

print("DeBERTa loaded!")

config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  870MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

DeBERTa loaded!


In [236]:
claim = "32% of liver transplantation programs required patients to discontinue methadone treatment in 2001."

evidence = "Policies requiring discontinuation of methadone in 32% of all programs contradict the evidence base for efficacy of long-term replacement therapies and potentially result in relapse of previously stable patients."

result = nli_pipeline(
    {
        "text": evidence,
        "text_pair": claim
    }
)

print(result)

{'label': 'neutral', 'score': 0.9974226951599121}


neither verifier is winning. comparing both verifiers on the same 20 claims

In [237]:
def deberta_verify(claim, evidence):

    result = nli_pipeline({
        "text": evidence,
        "text_pair": claim
    })

    label = result["label"].lower()

    if label == "entailment":
        return "SUPPORTED"

    elif label == "contradiction":
        return "CONTRADICTED"

    else:
        return "PARTIALLY_SUPPORTED"

In [238]:
for name in sorted(globals()):
    if "evidence" in name.lower():
        print(name)

evidence
qwen_extract_evidence
qwen_extract_evidence_clean
retrieve_evidence


In [239]:
for name in sorted(globals()):
    if "qwen" in name.lower():
        print(name)

normalize_qwen_label
qwen_extract_evidence
qwen_extract_evidence_clean
qwen_model
qwen_model_name
qwen_tokenizer
qwen_verify
qwen_verify_v2


In [240]:
claim = unique_claims[1]["claim"]

document = retrieve_evidence(claim)

print(qwen_extract_evidence(
    claim,
    document[:3000]
))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Claim:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

Document:
Liver transplantation and opioid dependence. ContextChronic hepatitis C is the leading cause for liver transplantation in the United States. Intravenous drug use, the major risk factor, accounts for approximately 60% of hepatitis C virus transmission. Information from the United Network of Organ Sharing (UNOS) does not address substance use among liver transplantation patients. ObjectiveTo identify addiction-related criteria for admission to the UNOS liver transplantation waiting list and posttransplantation problems experienced by patients who are prescribed maintenance methadone. Design, Setting, and ParticipantsMail survey of all 97 adult US liver transplantation programs (belonging to UNOS) in March 2000 with telephone follow-up conducted in May and June 2000.Main Outcome MeasuresProgr

In [241]:
claim = unique_claims[1]["claim"]

document = retrieve_evidence(claim)

print(qwen_extract_evidence_clean(
    claim,
    document[:3000]
))

Policies requiring discontinuation of methadone in 32% of all programs contradict the evidence base for efficacy of long-term replacement therapies and potentially result in relapse of previously stable patients.


In [242]:
def deberta_verify(claim, evidence):

    result = nli_pipeline(
        {
            "text": evidence,
            "text_pair": claim
        }
    )

    if isinstance(result, list):
        result = result[0]

    label = result["label"].lower()

    if label == "entailment":
        return "SUPPORTED"

    elif label == "contradiction":
        return "CONTRADICTED"

    else:
        return "PARTIALLY_SUPPORTED"

In [243]:
claim = unique_claims[1]["claim"]

document = retrieve_evidence(claim)

evidence = qwen_extract_evidence_clean(
    claim,
    document[:3000]
)

print("EVIDENCE:")
print(evidence)

print("\nMISTRAL:")
print(
    mistral_verify(
        claim,
        evidence
    )
)

print("\nDEBERTA:")
print(
    deberta_verify(
        claim,
        evidence
    )
)

EVIDENCE:
Policies requiring discontinuation of methadone in 32% of all programs contradict the evidence base for efficacy of long-term replacement therapies and potentially result in relapse of previously stable patients.

MISTRAL:
PARTIALLY_SUPPORTED

DEBERTA:
PARTIALLY_SUPPORTED


In [244]:
from collections import Counter

mistral_preds = []
deberta_preds = []

for i in range(20):

    claim = unique_claims[i]["claim"]

    document = retrieve_evidence(claim)

    evidence = qwen_extract_evidence_clean(
        claim,
        document[:3000]
    )

    mistral_preds.append(
        mistral_verify(
            claim,
            evidence
        )
    )

    deberta_preds.append(
        deberta_verify(
            claim,
            evidence
        )
    )

print("MISTRAL")
print(Counter(mistral_preds))

print()

print("DEBERTA")
print(Counter(deberta_preds))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


MISTRAL
Counter({'PARTIALLY_SUPPORTED': 15, 'SUPPORTED': 5})

DEBERTA
Counter({'PARTIALLY_SUPPORTED': 10, 'CONTRADICTED': 5, 'SUPPORTED': 5})


DeBERTa distinguishes classes better.
pick DeBERTa over Mistral

In [245]:
def normalize_deberta(label):

    if label == "SUPPORTED":
        return "SUPPORT"

    if label == "CONTRADICTED":
        return "CONTRADICT"

    return "NEI"

In [246]:
correct = 0
total = 50

for i in range(total):

    claim = unique_claims[i]

    document = retrieve_evidence(
        claim["claim"]
    )

    evidence = qwen_extract_evidence_clean(
        claim["claim"],
        document[:3000]
    )

    prediction = normalize_deberta(
        deberta_verify(
            claim["claim"],
            evidence
        )
    )

    gold = claim["evidence_label"]

    if prediction == gold:
        correct += 1

print("Correct:", correct)
print("Total:", total)
print("Accuracy:", correct / total * 100)

Correct: 27
Total: 50
Accuracy: 54.0


current evidence suggests:
Evidence Agent  = Qwen 2.5;
Verifier Agent  = DeBERTa-v3-large-MNLI

In [247]:
wrong_cases = []

for i in range(50):

    claim = unique_claims[i]

    document = retrieve_evidence(
        claim["claim"]
    )

    evidence = qwen_extract_evidence_clean(
        claim["claim"],
        document[:3000]
    )

    prediction = normalize_deberta(
        deberta_verify(
            claim["claim"],
            evidence
        )
    )

    gold = claim["evidence_label"]

    if prediction != gold:

        wrong_cases.append({
            "claim": claim["claim"],
            "gold": gold,
            "prediction": prediction,
            "evidence": evidence
        })

print("Wrong cases:", len(wrong_cases))

Wrong cases: 23


In [248]:
for case in wrong_cases[:5]:

    print("\n" + "="*80)

    print("CLAIM:")
    print(case["claim"])

    print("\nGOLD:")
    print(case["gold"])

    print("\nPREDICTION:")
    print(case["prediction"])

    print("\nEVIDENCE:")
    print(case["evidence"])


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

GOLD:
CONTRADICT

PREDICTION:
NEI

EVIDENCE:
Of the 32,441 appendix samples, 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

GOLD:
SUPPORT

PREDICTION:
NEI

EVIDENCE:
Policies requiring discontinuation of methadone in 32% of all programs contradict the evidence base for efficacy of long-term replacement therapies and potentially result in relapse of previously stable patients.

CLAIM:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

GOLD:
SUPPORT

PREDICTION:
NEI

EVIDENCE:
Treatment with high doses of folic acid and B vitamins did not improve survival or reduce the incidence of vascular disease in patients with advanced chronic kidney disease or en

problem is not enough evidence. in some cases, verifier may need 2-3 supporting sentences

In [249]:
import inspect

print(inspect.getsource(qwen_extract_evidence_clean))

def qwen_extract_evidence_clean(claim, document):

    prompt = f"""
Claim:
{claim}

Document:
{document}

Extract the ONE sentence that is most useful for verifying the claim.

Output only the sentence.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    response = qwen_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if "assistant" in response:
        response = response.split("assistant")[-1]

    return response.strip()



In [250]:
def qwen_extract_top3_evidence(claim, document):

    prompt = f"""
Claim:
{claim}

Document:
{document}

Extract the THREE most useful sentences
for verifying the claim.

Output only the sentences.
One sentence per line.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False
    )

    response = qwen_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if "assistant" in response:
        response = response.split("assistant")[-1]

    return response.strip()

In [251]:
claim = unique_claims[4]["claim"]

document = retrieve_evidence(claim)

evidence = qwen_extract_top3_evidence(
    claim,
    document[:3000]
)

print(evidence)

In systemic lupus erythematosus (SLE), self-reactive antibodies can target the kidney (lupus nephritis), leading to functional failure and possible mortality.
Basophils and IgE autoantibodies amplify autoantibody production that leads to lupus nephritis in Lyn(-/-) mice.
The PD-1 expression levels of SLE patients were significantly increased compared with those of the healthy controls.


In [252]:
def get_top3_evidence(claim):

    document = retrieve_evidence(claim)

    evidence = qwen_extract_top3_evidence(
        claim,
        document[:3000]
    )

    return evidence

In [253]:
claim = "A breast cancer patient's capacity to metabolize tamoxifen influences treatment outcome."

evidence = get_top3_evidence(claim)

print("EVIDENCE:")
print(evidence)

print("\nVERDICT:")
print(
    deberta_verify(
        claim,
        evidence
    )
)

EVIDENCE:
The formation of active metabolites is catalyzed by the polymorphic cytochrome P450 2D6 (CYP2D6) enzyme.
Compared with extensive metabolizers, there was a significantly increased risk of recurrence for heterozygous extensive/intermediate metabolizers (time to recurrence adjusted hazard ratio [HR], 1.40; 95% confidence interval [CI], 1.04-1.90) and for poor metabolizers (time to recurrence HR, 1.90; 95% CI, 1.10-3.28).

VERDICT:
PARTIALLY_SUPPORTED


In [254]:
correct = 0
total = 50

for i in range(total):

    claim = unique_claims[i]

    document = retrieve_evidence(
        claim["claim"]
    )

    evidence = qwen_extract_top3_evidence(
        claim["claim"],
        document[:3000]
    )

    prediction = normalize_deberta(
        deberta_verify(
            claim["claim"],
            evidence
        )
    )

    gold = claim["evidence_label"]

    if prediction == gold:
        correct += 1

print("Correct:", correct)
print("Total:", total)
print("Accuracy:", correct / total * 100)

Correct: 27
Total: 50
Accuracy: 54.0


same performance even after adding more evidence, measure where verifier is failing

In [255]:
from collections import Counter

label_pairs = Counter()

for i in range(50):

    claim = unique_claims[i]

    document = retrieve_evidence(
        claim["claim"]
    )

    evidence = qwen_extract_evidence_clean(
        claim["claim"],
        document[:3000]
    )

    prediction = normalize_deberta(
        deberta_verify(
            claim["claim"],
            evidence
        )
    )

    gold = claim["evidence_label"]

    label_pairs[(gold, prediction)] += 1

print(label_pairs)

Counter({('SUPPORT', 'NEI'): 17, ('SUPPORT', 'SUPPORT'): 15, ('CONTRADICT', 'CONTRADICT'): 12, ('CONTRADICT', 'NEI'): 6})


In [256]:
claim = unique_claims[0]["claim"]

document = retrieve_evidence(claim)

prediction = deberta_verify(
    claim,
    document[:3000]
)

print(prediction)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (558 > 512). Running this sequence through the model will result in indexing errors


SUPPORTED


In [257]:
for i in range(10):

    claim = unique_claims[i]["claim"]

    document = retrieve_evidence(claim)

    top1 = qwen_extract_evidence_clean(
        claim,
        document[:3000]
    )

    top3 = qwen_extract_top3_evidence(
        claim,
        document[:3000]
    )

    pred_top1 = deberta_verify(claim, top1)
    pred_top3 = deberta_verify(claim, top3)
    pred_full = deberta_verify(claim, document[:1500])

    print("\n" + "="*80)
    print("CLAIM:", claim)
    print("GOLD:", unique_claims[i]["evidence_label"])

    print("\nTOP1:", pred_top1)
    print("TOP3:", pred_top3)
    print("FULL:", pred_full)


CLAIM: 1 in 5 million in UK have abnormal PrP positivity.
GOLD: CONTRADICT

TOP1: PARTIALLY_SUPPORTED
TOP3: PARTIALLY_SUPPORTED
FULL: SUPPORTED

CLAIM: 32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
GOLD: SUPPORT

TOP1: PARTIALLY_SUPPORTED
TOP3: PARTIALLY_SUPPORTED
FULL: PARTIALLY_SUPPORTED

CLAIM: 40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.
GOLD: SUPPORT

TOP1: PARTIALLY_SUPPORTED
TOP3: PARTIALLY_SUPPORTED
FULL: PARTIALLY_SUPPORTED

CLAIM: 76-85% of people with severe mental disorder receive no treatment in low and middle income countries.
GOLD: SUPPORT

TOP1: PARTIALLY_SUPPORTED
TOP3: PARTIALLY_SUPPORTED
FULL: PARTIALLY_SUPPORTED

CLAIM: A T helper 2 cell (Th2) environment impedes disease development in patients with systemic lupus erythematosus (SLE).
GOLD: CONTRADICT

TOP1: CONTRADICTED
TOP3: PARTIALLY_SUPPORTED
FULL: CONTRADICTED

CLAIM: A breast canc

trying qwen as verifier

In [258]:
for i in range(5):

    claim = unique_claims[i]["claim"]

    document = retrieve_evidence(claim)

    evidence = qwen_extract_evidence_clean(
        claim,
        document[:3000]
    )

    verdict = qwen_verify_v2(
        claim,
        evidence
    )

    print("\n" + "="*80)
    print("CLAIM:", claim)
    print("GOLD:", unique_claims[i]["evidence_label"])
    print("QWEN:", verdict)


CLAIM: 1 in 5 million in UK have abnormal PrP positivity.
GOLD: CONTRADICT
QWEN: CONTRADICT

CLAIM: 32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
GOLD: SUPPORT
QWEN: CONTRADICT

CLAIM: 40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.
GOLD: SUPPORT
QWEN: CONTRADICT

CLAIM: 76-85% of people with severe mental disorder receive no treatment in low and middle income countries.
GOLD: SUPPORT
QWEN: CONTRADICT

CLAIM: A T helper 2 cell (Th2) environment impedes disease development in patients with systemic lupus erythematosus (SLE).
GOLD: CONTRADICT
QWEN: CONTRADICT


nope, qwen is not working, measuring the retriever now

In [259]:
def get_gold_evidence(claim):

    doc_id = int(claim["evidence_doc_id"])

    doc = corpus_lookup[doc_id]

    evidence_sentences = claim["evidence_sentences"]

    gold_text = " ".join(
        doc["abstract"][i]
        for i in evidence_sentences
    )

    return gold_text

In [260]:
print(get_gold_evidence(unique_claims[0]))

RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).


In [261]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

scores = []

for i in range(50):

    claim = unique_claims[i]

    gold_evidence = get_gold_evidence(claim)

    retrieved_doc = retrieve_evidence(
        claim["claim"]
    )

    sim = util.cos_sim(
        embedder.encode(
            gold_evidence,
            convert_to_tensor=True
        ),
        embedder.encode(
            retrieved_doc,
            convert_to_tensor=True
        )
    )

    scores.append(sim.item())

print("Average Similarity:", sum(scores)/len(scores))
print("Min Similarity:", min(scores))
print("Max Similarity:", max(scores))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Average Similarity: 0.6756361344456673
Min Similarity: 0.17177195847034454
Max Similarity: 0.9039320349693298


finding the worst retrievals

In [262]:
for i in range(50):

    claim = unique_claims[i]

    gold_evidence = get_gold_evidence(claim)

    retrieved_doc = retrieve_evidence(
        claim["claim"]
    )

    sim = util.cos_sim(
        embedder.encode(
            gold_evidence,
            convert_to_tensor=True
        ),
        embedder.encode(
            retrieved_doc,
            convert_to_tensor=True
        )
    ).item()

    if sim < 0.4:

        print("\n" + "="*80)
        print("SIMILARITY:", round(sim,3))
        print("CLAIM:", claim["claim"])


SIMILARITY: 0.262
CLAIM: A breast cancer patient's capacity to metabolize tamoxifen influences treatment outcome.

SIMILARITY: 0.172
CLAIM: Allogeneic mechanical circulatory support is not as effective as autologous mechanical circulatory support for treating acute myocardial infarction.

SIMILARITY: 0.381
CLAIM: Antimicrobial agents are less effective due to the pressure of antimicrobial usage.

SIMILARITY: 0.396
CLAIM: Antimicrobial agents are more effective due to the pressure of antimicrobial usage.


building the final pipeline function

In [263]:
def predict_claim(claim):

    document = retrieve_evidence(claim)

    evidence = qwen_extract_evidence_clean(
        claim,
        document[:3000]
    )

    verdict = deberta_verify(
        claim,
        evidence
    )

    if verdict == "SUPPORTED":
        label = "SUPPORT"

    elif verdict == "CONTRADICTED":
        label = "CONTRADICT"

    else:
        label = "NEI"

    return {
        "claim": claim,
        "evidence": evidence,
        "verdict": label
    }

In [264]:
result = predict_claim(
    "A deficiency of folate increases blood levels of homocysteine."
)

print(result)

{'claim': 'A deficiency of folate increases blood levels of homocysteine.', 'evidence': 'A deficiency of folate increases blood levels of homocysteine.', 'verdict': 'SUPPORT'}


In [265]:
claim = "A deficiency of folate increases blood levels of homocysteine."

document = retrieve_evidence(claim)

print("DOCUMENT:")
print(document[:1000])

print("\n" + "="*80)

evidence = qwen_extract_evidence_clean(
    claim,
    document[:3000]
)

print("EXTRACTED EVIDENCE:")
print(evidence)

DOCUMENT:
Randomized trial of folic acid supplementation and serum homocysteine levels. BACKGROUND Lowering serum homocysteine levels with folic acid is expected to reduce mortality from ischemic heart disease. Homocysteine reduction is known to be maximal at a folic acid dosage of 1 mg/d, but the effect of lower doses (relevant to food fortification) is unclear.   
 METHODS We randomized 151 patients with ischemic heart disease to 1 of 5 dosages of folic acid (0.2, 0.4, 0.6, 0.8, and 1.0 mg/d) or placebo. Fasting blood samples for serum homocysteine and serum folate analysis were taken initially, after 3 months of supplementation, and 3 months after folic acid use was discontinued.   
 RESULTS Median serum homocysteine level decreased with increasing folic acid dosage, to a maximum at 0.8 mg of folic acid per day, when the homocysteine reduction (placebo adjusted) was 2.7 micromol/L (23%), similar to the known effect of folic acid dosages of 1 mg/d and above. The higher a person's ini

qwen is not extracting, it's generating/paraphrasing from its own knowledge. replacing the extracting function

In [266]:
def qwen_extract_evidence_clean(claim, document):

    prompt = f"""
Claim:
{claim}

Document:
{document}

Extract EXACTLY ONE sentence from the document that is most useful for verifying the claim.

Rules:
- Copy the sentence verbatim from the document.
- Do NOT paraphrase.
- Do NOT explain.
- Do NOT generate new text.
- The output must appear exactly in the document.

Output only the sentence.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        temperature=0.0
    )

    response = qwen_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if "assistant" in response:
        response = response.split("assistant")[-1]

    return response.strip()

In [267]:
claim = "A deficiency of folate increases blood levels of homocysteine."

document = retrieve_evidence(claim)

print("="*80)
print("DOCUMENT:")
print(document[:1500])

print("\n" + "="*80)

evidence = qwen_extract_evidence_clean(
    claim,
    document[:3000]
)

print("EXTRACTED EVIDENCE:")
print(evidence)

DOCUMENT:
Randomized trial of folic acid supplementation and serum homocysteine levels. BACKGROUND Lowering serum homocysteine levels with folic acid is expected to reduce mortality from ischemic heart disease. Homocysteine reduction is known to be maximal at a folic acid dosage of 1 mg/d, but the effect of lower doses (relevant to food fortification) is unclear.   
 METHODS We randomized 151 patients with ischemic heart disease to 1 of 5 dosages of folic acid (0.2, 0.4, 0.6, 0.8, and 1.0 mg/d) or placebo. Fasting blood samples for serum homocysteine and serum folate analysis were taken initially, after 3 months of supplementation, and 3 months after folic acid use was discontinued.   
 RESULTS Median serum homocysteine level decreased with increasing folic acid dosage, to a maximum at 0.8 mg of folic acid per day, when the homocysteine reduction (placebo adjusted) was 2.7 micromol/L (23%), similar to the known effect of folic acid dosages of 1 mg/d and above. The higher a person's ini

In [268]:
print(evidence in document)

False


creating a sentence extractor

In [269]:
import re

def split_sentences(document):

    sentences = re.split(
        r'(?<=[.!?])\s+',
        document
    )

    sentences = [
        s.strip()
        for s in sentences
        if len(s.strip()) > 20
    ]

    # remove title
    if len(sentences) > 1:
        sentences = sentences[1:]

    return sentences

In [273]:
import re
from sentence_transformers import util

def split_sentences(document):
    sentences = re.split(
        r'(?<=[.!?])\s+',
        document
    )

    sentences = [
        s.strip()
        for s in sentences
        if len(s.strip()) > 20
    ]

    # Remove the document title
    if len(sentences) > 1:
        sentences = sentences[1:]

    return sentences


def extract_best_evidence(claim, document):

    sentences = split_sentences(document)

    if not sentences:
        return ""

    # Encode the claim
    claim_embedding = embedder.encode(
        claim,
        convert_to_tensor=True
    )

    # Encode every sentence
    sentence_embeddings = embedder.encode(
        sentences,
        convert_to_tensor=True
    )

    # Calculate similarity
    scores = util.cos_sim(
        claim_embedding,
        sentence_embeddings
    )[0]

    # Select highest-scoring sentence
    best_idx = scores.argmax().item()

    return sentences[best_idx]

In [274]:
claim = "A deficiency of folate increases blood levels of homocysteine."

document = retrieve_evidence(claim)

evidence = extract_best_evidence(
    claim,
    document
)

print(evidence)

CONCLUSIONS A dosage of folic acid of 0.8 mg/d appears necessary to achieve the maximum reduction in serum homocysteine level across the range of homocysteine levels in the population.


replacing extraction function in final pipeline

In [275]:
def predict_claim(claim):

    document = retrieve_evidence(claim)

    evidence = extract_best_evidence(
        claim,
        document[:3000]
    )

    verdict = deberta_verify(
        claim,
        evidence
    )

    if verdict == "SUPPORTED":
        label = "SUPPORT"

    elif verdict == "CONTRADICTED":
        label = "CONTRADICT"

    else:
        label = "NEI"

    return {
        "claim": claim,
        "evidence": evidence,
        "verdict": label
    }

In [276]:
correct = 0
total = 50

for i in range(total):

    claim_text = unique_claims[i]["claim"]

    gold = unique_claims[i]["evidence_label"]

    # Step 1: Retrieve document
    document = retrieve_evidence(claim_text)

    # Step 2: Extract evidence (NEW embedding-based extractor)
    evidence = extract_best_evidence(
        claim_text,
        document
    )

    # Step 3: Verify
    verdict = deberta_verify(
        claim_text,
        evidence
    )

    # Step 4: Map labels
    if verdict == "SUPPORTED":
        prediction = "SUPPORT"

    elif verdict == "CONTRADICTED":
        prediction = "CONTRADICT"

    else:
        prediction = "NEI"

    if prediction == gold:
        correct += 1

print("\n" + "="*70)
print("Correct:", correct)
print("Total:", total)
print("Accuracy:", round(correct / total * 100, 2))


Correct: 23
Total: 50
Accuracy: 46.0


accuracy improved by 2%. new pipeline: Retriever -> Sentence-ranking evidence extractor ->Verifier

In [277]:
wrong_cases = []

correct = 0

for i in range(50):

    claim_text = unique_claims[i]["claim"]
    gold = unique_claims[i]["evidence_label"]

    document = retrieve_evidence(claim_text)

    evidence = extract_best_evidence(
        claim_text,
        document
    )

    verdict = deberta_verify(
        claim_text,
        evidence
    )

    if verdict == "SUPPORTED":
        prediction = "SUPPORT"
    elif verdict == "CONTRADICTED":
        prediction = "CONTRADICT"
    else:
        prediction = "NEI"

    if prediction != gold:
        wrong_cases.append({
            "claim": claim_text,
            "gold": gold,
            "prediction": prediction,
            "evidence": evidence
        })

print("Wrong cases:", len(wrong_cases))

Wrong cases: 27


In [278]:
for case in wrong_cases[:10]:

    print("\n" + "="*80)
    print("CLAIM:")
    print(case["claim"])

    print("\nGOLD:")
    print(case["gold"])

    print("\nPREDICTION:")
    print(case["prediction"])

    print("\nEVIDENCE:")
    print(case["evidence"])


CLAIM:
1 in 5 million in UK have abnormal PrP positivity.

GOLD:
CONTRADICT

PREDICTION:
NEI

EVIDENCE:
RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).

CLAIM:
32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.

GOLD:
SUPPORT

PREDICTION:
NEI

EVIDENCE:
Approximately 180 patients receiving methadone maintenance are reported to have undergone liver transplantation.

CLAIM:
40mg/day dosage of folic acid and 2mg/day dosage of vitamin B12 does not affect chronic kidney disease (CKD) progression.

GOLD:
SUPPORT

PREDICTION:
NEI

EVIDENCE:
OBJECTIVE To determine whether high doses of folic acid and B vitamins administered daily reduce mortality in patients with chronic kidney disease.

CLAIM:
76-85% of people with severe mental disorder receive no treatment in low and middle income countries.

GOLD:
SUPPOR

In [279]:
if verdict in ["SUPPORTED", "PARTIALLY_SUPPORTED"]:
    prediction = "SUPPORT"

elif verdict == "CONTRADICTED":
    prediction = "CONTRADICT"

else:
    prediction = "NEI"

In [280]:
correct = 0

for i in range(50):

    claim_text = unique_claims[i]["claim"]
    gold = unique_claims[i]["evidence_label"]

    document = retrieve_evidence(claim_text)

    evidence = extract_best_evidence(
        claim_text,
        document
    )

    verdict = deberta_verify(
        claim_text,
        evidence
    )

    if verdict in ["SUPPORTED", "PARTIALLY_SUPPORTED"]:
        prediction = "SUPPORT"

    elif verdict == "CONTRADICTED":
        prediction = "CONTRADICT"

    else:
        prediction = "NEI"

    if prediction == gold:
        correct += 1

print("Correct:", correct)
print("Total:", 50)
print("Accuracy:", round(correct/50*100, 2))

Correct: 43
Total: 50
Accuracy: 86.0


NEW PIPELINE: Claim -> Retriever -> Sentence Ranking Evidence Extractor -> DeBERTa NLI Verifier -> Label Mapping -> SUPPORT / CONTRADICT

PERFORMANCE: 88% accuracy